# DIMER Notebook: Modern Image Classification and Transfer Representations
## From ResNet to MobileNet, ConvNeXt and Vision Transformers

**Notebook profile:** `E2E`  
**Pedagogical mode:** `WORKSHOP`  
**DIMER Notebook Specification:** `2.1`  
**Comparison scope:** `MULTI-MODEL`  
**Standalone:** yes  
**Canonical task:** six-species image classification  
**Adaptation:** frozen backbone + identical six-class linear probe

This notebook compares six live DIMER image-classification backbones:

1. ResNet-50
2. MobileNetV4-Conv-Small
3. ConvNeXt-Tiny
4. ViT-B/16
5. SwinV2-Tiny
6. EVA-02 Base 448

The native ImageNet heads are not compared. Every checkpoint is instead treated as a frozen feature extractor, then the exact same linear classifier is fitted on top of its representation.

> **Interpretation boundary:** this is a **pretrained checkpoint comparison**, not a pure architecture ablation. The models differ in architecture, pretraining data, training recipe, model capacity and native input resolution.

## How to use this notebook

**Who it is for.** Learners who can run Python cells in Colab/Jupyter and are new to modern vision backbones, frozen representations, or transfer learning.

**Runtime.** Use the documented GPU runtime for the six-backbone comparison.

**How to run it.**
1. Select the documented runtime/accelerator.
2. Choose **Run all** for the canonical path; leave the default settings unchanged on your first pass.
3. Read the explanatory markdown while the notebook runs.
4. Sections marked **Infrastructure** support reproducibility, model acquisition, or orchestration. Run those cells as written; understanding their implementation is not a learning objective.

### Task at a glance

`image → frozen pretrained backbone → representation → k-NN / common linear probe → predicted species`

### Roadmap

1. Understand the task and its input/output contract.
2. Inspect and validate the built-in data or inputs.
3. Establish the baseline/reference behavior.
4. Run the model or multi-model comparison.
5. Inspect errors, disagreements, robustness, and/or resource tradeoffs.
6. Try one controlled change and explain what changed.
7. Write an evidence-based conclusion; optionally continue with BYOD.

### What successful execution looks like

You should finish with a validated input/sample, the notebook's principal baseline/reference, model outputs and evaluation results, at least one diagnostic or qualitative comparison, and machine-readable results/provenance where supported. Exact values can vary slightly across supported runtimes; focus on the defined metrics and the observed pattern.


## 1. Why not compare the native ImageNet heads?

The built-in heads all predict ImageNet classes, while this notebook uses six bird species outside a clean shared six-class mapping.

A direct comparison would mix model quality with an arbitrary ontology-mapping policy.

The controlled experiment is instead:

`image → frozen pretrained representation → identical six-class linear probe`

## 2. Architectural progression

- **ResNet-50:** residual convolutional network.
- **MobileNetV4:** efficiency-oriented mobile convolutional network.
- **ConvNeXt-Tiny:** modernized ConvNet informed by transformer-era design choices.
- **ViT-B/16:** global self-attention over fixed image patches.
- **SwinV2-Tiny:** hierarchical transformer with local shifted-window attention.
- **EVA-02 Base:** high-capacity transformer with masked-image-modeling lineage and a 448×448 input.

The final comparison therefore spans both architectural ideas and practical deployment envelopes.

## 3. Configuration

The defaults define the canonical `Run all` path.

In [1]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ""  # @param {type:"string"}
BYOD_MODEL_KEYS = ["resnet50","mobilenetv4","convnext","vit","swinv2","eva02"]

FEATURE_BATCH_SIZE = 8  # @param {type:"integer"}
LATENCY_REPEATS = 10  # @param {type:"integer"}

# Full-batch AdamW from a zero-initialised linear layer. At lr 0.05 the lowest validation loss fell at epoch 1 for
# five of the six backbones (a one-step probe); lr 0.001 over 1000 epochs trains a real probe, as in the ViT workshop.
PROBE_EPOCHS = 1000
PROBE_LR = 0.001
PROBE_WEIGHT_DECAY = 1e-4
KNN_K = 5

RUN_DATA_EFFICIENCY = True  # @param {type:"boolean"}
DATA_EFFICIENCY_PER_CLASS = [6, 12, 18]

RUN_PCA_VISUALIZATION = True  # @param {type:"boolean"}
RUN_RESOLUTION_STRESS = False  # @param {type:"boolean"}

SAMPLE_SEED = 42
OUTPUT_DIR = "outputs/modern_image_classification"

if FEATURE_BATCH_SIZE < 1:
    raise ValueError("FEATURE_BATCH_SIZE must be >=1")
if LATENCY_REPEATS < 3:
    raise ValueError("LATENCY_REPEATS must be >=3")
print({
    "feature_batch_size": FEATURE_BATCH_SIZE,
    "latency_repeats": LATENCY_REPEATS,
    "data_efficiency": RUN_DATA_EFFICIENCY,
    "pca": RUN_PCA_VISUALIZATION,
})

{'feature_batch_size': 8, 'latency_repeats': 10, 'data_efficiency': True, 'pca': True}


## 4. Runtime

The notebook uses one Python 3.12 `timm` runtime for all six backbones. Models are loaded sequentially and never remain resident together.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [2]:
import importlib.metadata as importlib_metadata
import subprocess
import sys

PINS = {
    "torch": "2.14.0",
    "torchvision": "0.29.0",
    "torchaudio": "2.11.0",
    "timm": "1.0.29",
    "safetensors": "0.8.0",
    "numpy": "2.5.3",
    "pillow": "11.3.0",
    "huggingface-hub": "0.36.2",
    "matplotlib": "3.10.6",
}

def dist_version(name):
    """The public version (PEP 440 without a local label such as +cu130), as pip compares `==` pins."""
    try:
        return importlib_metadata.version(name).split("+", 1)[0]
    except importlib_metadata.PackageNotFoundError:
        return None

# Hosted kernels such as Colab import NumPy at startup. Replacing a loaded module on disk would force a manual
# restart (Notebook Spec RUN10), so a NumPy 2.x that is already loaded is kept and recorded instead of reinstalled.
NUMPY_PRELOADED = None
if "numpy" in sys.modules and str(getattr(sys.modules["numpy"], "__version__", "")).startswith("2."):
    NUMPY_PRELOADED = sys.modules["numpy"].__version__
    PINS["numpy"] = NUMPY_PRELOADED

before = {k:dist_version(k) for k in PINS}
needed = [f"{k}=={v}" for k,v in PINS.items() if before[k] != v]
if needed:
    print("Installing pinned runtime:", needed)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *needed])

after = {k:dist_version(k) for k in PINS}
bad = {k:(after[k],v) for k,v in PINS.items() if after[k] != v}
if bad:
    raise RuntimeError(f"Pinned installation did not converge: {bad}")

stale = []
for module_name, dist_name in [("torch", "torch"), ("torchvision", "torchvision"), ("timm", "timm"), ("numpy", "numpy")]:
    module = sys.modules.get(module_name)
    if module is not None:
        runtime_v = getattr(module, "__version__", None)
        if runtime_v and str(runtime_v).split("+", 1)[0] != str(after[dist_name]):
            stale.append((module_name, runtime_v, after[dist_name]))
if stale:
    raise RuntimeError(
        "The kernel had already imported packages that the pinned install replaced on disk. "
        "Restart the Python session and choose Run all again (Colab: Runtime > Restart session; "
        f"do not delete the runtime, which discards the installed pins). Stale: {stale}"
    )

import numpy as np
import torch
import torch.nn.functional as F
import timm
import matplotlib.pyplot as plt
from PIL import Image
from safetensors.torch import load_file, save_file

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

RUNTIME = {
    "python":sys.version.split()[0],
    "torch":torch.__version__,
    "torchvision":importlib_metadata.version("torchvision"),
    "timm":timm.__version__,
    "safetensors":importlib_metadata.version("safetensors"),
    "numpy":np.__version__,
    "numpy_source":"preloaded by the host kernel" if NUMPY_PRELOADED else "pinned install",
    "pillow":importlib_metadata.version("pillow"),
    "device":DEVICE,
    "cuda_available":torch.cuda.is_available(),
}
if torch.cuda.is_available():
    RUNTIME["gpu_name"] = torch.cuda.get_device_name(0)
    RUNTIME["gpu_total_memory_bytes"] = torch.cuda.get_device_properties(0).total_memory
print(RUNTIME)

Installing pinned runtime: ['torch==2.14.0', 'torchvision==0.29.0', 'huggingface-hub==0.36.2', 'matplotlib==3.10.6']
{'python': '3.13.15', 'torch': '2.14.0+cu130', 'torchvision': '0.29.0', 'timm': '1.0.29', 'safetensors': '0.8.0', 'numpy': '2.1.3', 'numpy_source': 'preloaded by the host kernel', 'pillow': '11.3.0', 'device': 'cuda:0', 'cuda_available': True, 'gpu_name': 'Tesla T4', 'gpu_total_memory_bytes': 15637086208}


## 5. Dataset provenance

The dataset is the same pinned sample used by the live DIMER ViT carrier:

**iNaturalist CC0 bird photographs — six species**

- 180 photographs;
- 30 per species;
- CC0 1.0;
- selected a priori on 2026-09-19;
- each photo pinned by byte size and SHA-256;
- fetched at runtime from the iNaturalist open-data bucket;
- no image redistributed by this notebook.

The six species are visually similar enough to make feature transfer non-trivial.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [3]:
import csv, hashlib, io, json, random, time, urllib.request, zipfile, gc
from collections import Counter, defaultdict
from pathlib import Path

SAMPLE_ROWS = json.loads(r'''[["song_sparrow-00","song_sparrow",129376982,79016324,"andywilson",43427,"7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d","jpg"],["song_sparrow-01","song_sparrow",480991086,267636534,"lyneisfilm",162073,"11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec","jpg"],["song_sparrow-02","song_sparrow",546060381,302980489,"swpollinators",27899,"4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6","jpg"],["song_sparrow-03","song_sparrow",308625896,177450028,"radrat",70961,"1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5","jpeg"],["song_sparrow-04","song_sparrow",494793016,275349085,"k-simpkins",58410,"255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e","jpg"],["song_sparrow-05","song_sparrow",674054489,369029444,"ben142",220573,"1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123","jpg"],["song_sparrow-06","song_sparrow",339623726,193339933,"rawcomposition",25012,"d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b","jpg"],["song_sparrow-07","song_sparrow",222768957,130949329,"davidfbird",110773,"0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff","jpg"],["song_sparrow-08","song_sparrow",181658744,107953669,"gcart043",98482,"a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4","jpeg"],["song_sparrow-09","song_sparrow",148994242,90171417,"glennberry",102702,"64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa","jpg"],["song_sparrow-10","song_sparrow",637315932,349374463,"sooji",136572,"a5cebbc0cc2325d3805c4ac854e103f7ba1c22e3a34fb204f30d58681e865033","jpg"],["song_sparrow-11","song_sparrow",471146686,262252507,"jeanpaulboerekamps",98601,"4301f06b52b8dcf1e137567c32412e384edb71cb90e2e35459a0405b2e56529b","jpg"],["song_sparrow-12","song_sparrow",123859010,75689904,"w_mark_c",190819,"6b6057a1c50b83ffeb4d9e34367b3e6a9b355236dbcae9820b509e1484f8fe33","jpg"],["song_sparrow-13","song_sparrow",640811426,351179648,"erikschiff",107695,"27aecce184a485ee888c0e8101cb4a34899b8a185b62dc064f3dc2ec9902182b","jpg"],["song_sparrow-14","song_sparrow",63537800,40010230,"nathanael15",51441,"5ae55f868e779a4e8ee34f6ad077b40c41f20aa699d967f373ebd659a89137a8","jpg"],["song_sparrow-15","song_sparrow",108520869,67204020,"dugald",52496,"e483364889fb95c540db84b5edf5a2400febf7d2eb62a1e49303b13ad329d1b8","jpg"],["song_sparrow-16","song_sparrow",435251292,244042351,"carterdorscht",166662,"d0cdd9ddcf7202a91ac2c47910a0c230639bce50283e8511fb8311151763233a","jpeg"],["song_sparrow-17","song_sparrow",120710033,73898230,"tys_rbg",126820,"515b3b32b64e88d401990bfd1d8e2c1281443b5d1e763a09e86d3d3664e1df41","jpg"],["song_sparrow-18","song_sparrow",393408927,222067370,"irenemacaulay_",101681,"22e326807de963352b4307790bd40c5506adb0fab1f9844968edaf4bc5665e1c","jpg"],["song_sparrow-19","song_sparrow",349361283,198243065,"sean579",42820,"5eb0031a8d6066650c66b265fb1724413273e095e4e531054d2817eb75910074","jpg"],["song_sparrow-20","song_sparrow",608802621,335154467,"jamesadney",112564,"6e94bce1b5135f48b0b19b64e21a0c76cff4f9b9a28fcbc2b7f5c8ddda6ef513","jpg"],["song_sparrow-21","song_sparrow",614258628,337847585,"joy4birds",70552,"aa767aa5a74a76cfd985aba5282e589f670a9ce0a8fcdc6a096c0357992dec67","jpg"],["song_sparrow-22","song_sparrow",634100352,347744524,"zorthesosen",214044,"6a70dba26dfb3e98a244a9ec9badb812ddaa8b3612e7a36f66aba057bb757ecd","jpg"],["song_sparrow-23","song_sparrow",50065474,31954532,"truthseqr",82521,"4186fbf344e92038358d4338102aa440098bff5f558ab1d197f72fefbcec0b15","jpeg"],["song_sparrow-24","song_sparrow",8656044,6803564,"glmory",297964,"e8cab773436ccfaad4699e5112ef4edb516236d413d6dbf034eea7bf9c88c8f5","jpg"],["song_sparrow-25","song_sparrow",131051149,79988590,"funvill",72572,"0a694b5bb03aee6eb5a3a6132e855d47172b21b84cc4cc7c64367d1a31de7894","jpeg"],["song_sparrow-26","song_sparrow",12923156,9491600,"gambolingquail",69559,"6a92bff4fc76820c6f21e15c5f254385dd976a32264911bfc08417a1c2bf053d","jpg"],["song_sparrow-27","song_sparrow",12077500,8959545,"reuvenm",74010,"3e3c5e94c839f45610ef3ef7ffaf3575f4bfe365fc94454c30bfdb718ca3c593","jpeg"],["song_sparrow-28","song_sparrow",132730299,80955309,"steph123456",154309,"5f8854dd231a302c643b22521b49ed3007203d74b8b8413a23abc79ffe0b0270","jpeg"],["song_sparrow-29","song_sparrow",124840110,76316566,"terrimewbornagain",81046,"2020092b0b67397a78cc2b0df267fb671b3ba18e03fb0e35e664e0784061e3c6","jpeg"],["chipping_sparrow-00","chipping_sparrow",198992636,117809422,"k-simpkins",62563,"cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf","jpg"],["chipping_sparrow-01","chipping_sparrow",248210057,144599194,"w_mark_c",193389,"497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150","jpg"],["chipping_sparrow-02","chipping_sparrow",156350853,94266719,"ellyne",142332,"6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2","jpeg"],["chipping_sparrow-03","chipping_sparrow",16128796,11327134,"reuvenm",70532,"5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c","jpeg"],["chipping_sparrow-04","chipping_sparrow",391300648,220982684,"carterdorscht",208567,"c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21","jpeg"],["chipping_sparrow-05","chipping_sparrow",339456083,193252042,"rawcomposition",46329,"41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496","jpg"],["chipping_sparrow-06","chipping_sparrow",40457652,26076708,"andywilson",156894,"b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6","jpeg"],["chipping_sparrow-07","chipping_sparrow",84151914,52921135,"davidfbird",145714,"ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9","jpeg"],["chipping_sparrow-08","chipping_sparrow",523674610,291074747,"rwp84",47983,"41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800","jpg"],["chipping_sparrow-09","chipping_sparrow",292695018,168861389,"tim_kirsten",64812,"cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d","jpeg"],["chipping_sparrow-10","chipping_sparrow",478480946,266314208,"russnamitz",61359,"8200cc2ec779d47be1b5afa261d934910a251a39f5620d3dc705106fb2bf5e0b","jpeg"],["chipping_sparrow-11","chipping_sparrow",220257388,129611350,"gcart043",129377,"1bdce32e7ac58321dd6beacc084afaf45973469215d66b61a0f7c612da3db361","jpeg"],["chipping_sparrow-12","chipping_sparrow",92501489,57964052,"tniernberger",78380,"44ddb9923026a97ee77ed362bc944c3d2cbbe4fbd4636000880e81613634e6c5","jpg"],["chipping_sparrow-13","chipping_sparrow",300376697,172991803,"matthias55",81986,"7a9a5cbfb6a7d0581c98c75b5fdeb53a049d60f352eda43a39b1f9c2a3347f3b","jpeg"],["chipping_sparrow-14","chipping_sparrow",58192408,36778771,"bradenjudson",22352,"7c630d7a5b24d94e677e563a8ecb36f57fb29babe6ac1568f11537f10a5edf75","jpeg"],["chipping_sparrow-15","chipping_sparrow",80410971,50636049,"radrat",55646,"0c71ca735502e8c94db81302ecd006428206eb16d75472eb0c706e62e2656667","jpg"],["chipping_sparrow-16","chipping_sparrow",538480223,298914072,"hiltonward",201271,"2a6479556f14a20a8c9a69ff0c1026fe4deb376944234d1d4429c558042c4346","jpg"],["chipping_sparrow-17","chipping_sparrow",370917657,209626382,"craigmartin",101736,"feb94d319fe5b12c01e63a80dc8e44c45e15a0bfb6534ac5d2c0641205dfe1d5","jpg"],["chipping_sparrow-18","chipping_sparrow",264119989,152960401,"laurelthrone",194107,"13e7336628f9ad784757dca82557c79eda22fdce561e0b52a10a503e72979b8e","jpeg"],["chipping_sparrow-19","chipping_sparrow",220323755,129631264,"enspring",85687,"886b7e6faa39943f0c9754e4aef637e2a2f5ad57fe8e8c7ec08360ea95af45b1","jpeg"],["chipping_sparrow-20","chipping_sparrow",210319996,124105091,"bunnymom20",188328,"2f6b6f7ac5ed9d91a361c8fed102484f4fca599ec39c9a2bfe9af1df53b28899","jpeg"],["chipping_sparrow-21","chipping_sparrow",99819947,62311494,"andy71",350329,"d649c3c9fd6b0b159a979572b48daba39fc5608104f21c6d88c7d10fa2479f7e","jpg"],["chipping_sparrow-22","chipping_sparrow",480169680,267207764,"cvharris",144339,"d1ab8643c48f2ea823f9b61843016795def21b290c35ce3d6045e4933b5dfa0a","jpeg"],["chipping_sparrow-23","chipping_sparrow",323624945,185313750,"mrspteranodon",232126,"dd60e466086350ce1b9f7a9ba7784fc3963b3f996326ccf52f5adffa5719c39d","jpeg"],["chipping_sparrow-24","chipping_sparrow",343116928,195090930,"umamimomma",52983,"a8948a0189d19b3d7b8df65271f4844b14bd5118f510d0b9ef458c942ac41ce2","jpg"],["chipping_sparrow-25","chipping_sparrow",354617430,200940828,"wafflemaster135",58919,"2ba9557d06a7f1bcb1c20908efc82ea6317e5a0bd0c858898f3b5c0a007d20fb","jpeg"],["chipping_sparrow-26","chipping_sparrow",352953833,200088501,"aster-asti",105786,"82c35e04e4e22fe35c9b364bce7737fa5226cb5c469c761adfc61ef0956da9db","jpg"],["chipping_sparrow-27","chipping_sparrow",465554743,259357260,"perrydise_koisplash",171532,"35c6469fad62f198c060e056bd298f87976b39055cf66601a265ab55d5562642","jpeg"],["chipping_sparrow-28","chipping_sparrow",148831027,90084486,"ian-wolfe",181942,"cfab51f0c0598120f5312b354ae249bafb15124dd74723a8c29cdd7319206e49","jpg"],["chipping_sparrow-29","chipping_sparrow",192945976,114256514,"sooji",200440,"04fd7d05d5ced3073d4c7ce8a4f659c6994bb485c5253b26fbc471452b4a0b14","jpeg"],["white_throated_sparrow-00","white_throated_sparrow",339621218,193338380,"rawcomposition",31152,"d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6","jpg"],["white_throated_sparrow-01","white_throated_sparrow",166821399,99992799,"dziakj1",125954,"c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852","jpeg"],["white_throated_sparrow-02","white_throated_sparrow",469820434,261505977,"joy4birds",111767,"7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb","jpg"],["white_throated_sparrow-03","white_throated_sparrow",99351488,62040646,"bradenjudson",23399,"e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e","jpeg"],["white_throated_sparrow-04","white_throated_sparrow",177497964,105746665,"andywilson",29827,"65475d4842396f2488167453192d4aa834d2f1b40bcc78b420878c55ccf694d7","jpeg"],["white_throated_sparrow-05","white_throated_sparrow",628148203,344731686,"lavenderdame",106872,"36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8","jpg"],["white_throated_sparrow-06","white_throated_sparrow",104660609,65043951,"allan7",42443,"10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635","jpeg"],["white_throated_sparrow-07","white_throated_sparrow",250718938,145903421,"stevestevens",138735,"bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4","jpeg"],["white_throated_sparrow-08","white_throated_sparrow",15105971,10793852,"schylerbrown",31467,"6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2","jpeg"],["white_throated_sparrow-09","white_throated_sparrow",171460784,102554447,"w_mark_c",251287,"3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137","jpg"],["white_throated_sparrow-10","white_throated_sparrow",244260317,142471526,"deejay",75694,"d67efb1ec61f6700b8a6c6552e2da9cd981e68ae81af16d6f1e0ef17fcaff84f","jpeg"],["white_throated_sparrow-11","white_throated_sparrow",194732675,115373159,"wildreturn",128989,"7ed4bde480b35734576bb4c5f9d1453e77c1f095380432b43b1b682050255831","jpeg"],["white_throated_sparrow-12","white_throated_sparrow",341990462,194541980,"ethologist",149260,"3e2710fac082cc347e6cd114d42d39f25ec47b82c25936922232eba916808cdb","jpg"],["white_throated_sparrow-13","white_throated_sparrow",267625208,154862375,"laurelthrone",148616,"0cac1bc7053c891ce5ae1c33e3f94b69f1b19b958bd08c686db2ccaa7ba0fbd1","jpeg"],["white_throated_sparrow-14","white_throated_sparrow",330539586,188793537,"efalquet",119214,"c3db4583124472b28dbfb4c6fd9b3829e451d508a6b0a095eccc28f17e7982b4","jpeg"],["white_throated_sparrow-15","white_throated_sparrow",193091403,114346779,"ian-wolfe",73751,"e28974c8782fccb00f5ea5420140e01248ead859f151563a95a26bf67acba479","jpg"],["white_throated_sparrow-16","white_throated_sparrow",691050773,377873006,"dinomariobob",167744,"cc5e25afa5041ce2d6ea4e0d726793843f3a867f30b8d9ccf55892d6617da887","jpg"],["white_throated_sparrow-17","white_throated_sparrow",440986574,246671481,"suzannehale",141857,"4a354c189225de2e7b4e94a6df9cbd3a671dac0c8a7d2d4c75e933f93d8b83bf","jpeg"],["white_throated_sparrow-18","white_throated_sparrow",113217820,69733707,"kemper",97802,"4de7da3ac53a086f2c343555be5a2b62a683715db9f80636a76673cc380fa7f6","jpeg"],["white_throated_sparrow-19","white_throated_sparrow",575307217,318327478,"k-simpkins",90416,"af2b4e72a098bd90b920c8a49632e1bbff18b73954d8ac541ca81ed4baf0530b","jpg"],["white_throated_sparrow-20","white_throated_sparrow",340420076,193765555,"don54",62714,"7f6eea434fc700166af1d343951d15f3a9c83eff06cfb0518c3c4291019a5556","jpeg"],["white_throated_sparrow-21","white_throated_sparrow",562344160,311510652,"rrfc",45842,"f7c6394649765db6e3313a03ae013291db1ae20d89d3b8f0bdf6641757f31ed0","jpg"],["white_throated_sparrow-22","white_throated_sparrow",74598266,47039878,"ianrwhyte",141502,"b0633999572a4499a80310955ab928a86f4fe08774e669b4c4949cd26254428b","jpeg"],["white_throated_sparrow-23","white_throated_sparrow",588001234,324825124,"portablecity",370187,"11e3a97f43de5d61266d15028fe9023eef3900cfea2d027bd94ad847ecba9607","jpg"],["white_throated_sparrow-24","white_throated_sparrow",694103481,379467387,"memoosborne",120656,"82684f82f62f6e036e206eec315fff8e1ba36d308b667eba06750b75d1fada3a","jpg"],["white_throated_sparrow-25","white_throated_sparrow",655749108,359408087,"russnamitz",58449,"2a7d2b4226f1941d0950adc04bfd8fe62345f1138c521d0f5f0adf8bc5838da1","jpg"],["white_throated_sparrow-26","white_throated_sparrow",599110899,330395670,"jd_flores",121343,"5eeef4f5533a4b0a0222dcf7be000a2b34f3331f3be2b939703155ba8c8ba3ae","jpg"],["white_throated_sparrow-27","white_throated_sparrow",653888173,358443775,"toknowtheland",144197,"20a4933b00967fe4488296b2b6e89c12ecbcca0e40da591bd95927cca46ae2f7","jpg"],["white_throated_sparrow-28","white_throated_sparrow",295037357,170132018,"sturuss",117077,"d309f75ad5f4008c906ba7b3d6138aa017f919a0a17a1d5b431fa7ab47d5f2d3","jpg"],["white_throated_sparrow-29","white_throated_sparrow",405042885,228270930,"carterdorscht",122157,"f1fb32bcfb78f66f50ccd20f2418832afe72c7a5847bdeb8e4dc19546455cfaf","jpeg"],["dark_eyed_junco-00","dark_eyed_junco",172110799,102901486,"schylerbrown",182973,"185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9","jpeg"],["dark_eyed_junco-01","dark_eyed_junco",46691943,29901256,"haida_gwaii",46823,"a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5","jpg"],["dark_eyed_junco-02","dark_eyed_junco",707222551,386266764,"ben142",289608,"7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89","jpg"],["dark_eyed_junco-03","dark_eyed_junco",192557376,114006980,"k-simpkins",172398,"206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479","jpeg"],["dark_eyed_junco-04","dark_eyed_junco",8793471,6892999,"truthseqr",45302,"f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2","jpeg"],["dark_eyed_junco-05","dark_eyed_junco",346777340,196961623,"zacharyfoster",46712,"ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab","jpg"],["dark_eyed_junco-06","dark_eyed_junco",274980085,159160633,"andy71",51209,"c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b","jpeg"],["dark_eyed_junco-07","dark_eyed_junco",243303909,141959574,"andywilson",71321,"ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6","jpeg"],["dark_eyed_junco-08","dark_eyed_junco",213798376,126031618,"nathanael15",57646,"d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b","jpg"],["dark_eyed_junco-09","dark_eyed_junco",63482066,39977347,"chrisleearm",41870,"90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72","jpeg"],["dark_eyed_junco-10","dark_eyed_junco",458710472,255914048,"joy4birds",90973,"fd25ecb1bf86896b84e9e4e8329c742011011e6883631f47dbe3744ba7463153","jpg"],["dark_eyed_junco-11","dark_eyed_junco",20784016,14046286,"gambolingquail",93957,"a454d6f987153317c9c57c05019b08ab0ebebc46d3cf72356b014231b69a1e9d","jpeg"],["dark_eyed_junco-12","dark_eyed_junco",332842159,189933284,"jan-konilu",110145,"d18706f536164a72a4ec9bacf47437edd89d6b547f86eefdf10ed0166cebf0dd","jpeg"],["dark_eyed_junco-13","dark_eyed_junco",513004508,270404136,"thevertebratepokedex",141252,"854f75e7527932d0409e14756725388de9913c0b450536da717e54d683de169f","jpg"],["dark_eyed_junco-14","dark_eyed_junco",593499256,327583635,"orionid",108583,"7eab4cea6d46faa878732d82c5c2ece63ecc3e66781ea08d0bd950eee581d16d","jpg"],["dark_eyed_junco-15","dark_eyed_junco",63590391,40041059,"bobbyblackmore",82853,"87746be0bca30fc40ddba739c1a35ed1fdd41882ebfe3561518b69d285f5ae5d","jpeg"],["dark_eyed_junco-16","dark_eyed_junco",12580989,9282523,"artemis224",216253,"15198c123c01fa0f8c03edc086408c8d95ffa553083c380e8ae236dbb7056093","jpg"],["dark_eyed_junco-17","dark_eyed_junco",12533281,9255403,"braincellsgone",40857,"be34c51655f59612858d888b05a4f927da65e1aede850c9a5f1be68fa00bfcbe","jpg"],["dark_eyed_junco-18","dark_eyed_junco",256948665,149140989,"igor322",86925,"9339b8cf4aa347b4eb176ef1209dfaa4fcb11e277889cc17de399d6af2c8633a","jpeg"],["dark_eyed_junco-19","dark_eyed_junco",106718005,66230973,"vicki936",41475,"e6903e9a69e987c45edd468ace0bf71adf8252063ff5d39c2130409d8fea68be","jpeg"],["dark_eyed_junco-20","dark_eyed_junco",611049594,336277421,"skylar_schell",15425,"7e876479febb3d44b1e494a25f778efe4e8bfd323279031cb08f9f6fe95f742e","jpg"],["dark_eyed_junco-21","dark_eyed_junco",591147962,326403963,"toknowtheland",106373,"07770f3313dc969802355fa8b3e62a86edb115a64771bd38938e1fab33220c39","jpg"],["dark_eyed_junco-22","dark_eyed_junco",469746672,261469406,"shannon_j",74167,"ec4c7ce15aa1c4eadd3bc5d46d433baf9560d6a0e4593d8d57aab8fa9bec9e18","jpg"],["dark_eyed_junco-23","dark_eyed_junco",459696953,256398190,"w_mark_c",262126,"5025e757c026e02160ff3f0e82f1f4434b888a6c9689e7ac76274c1fcec64f0c","jpg"],["dark_eyed_junco-24","dark_eyed_junco",484171775,269292238,"aschuman",59459,"89dd933d12cf21c0e73eb01e96a9279e2e57b36676a72a9c860281d0d7703282","jpg"],["dark_eyed_junco-25","dark_eyed_junco",357382685,202362003,"dougbrown",56324,"70275d8e07192c99e121b67d206de6823b194f043e65b9766f1f9ce772953c78","jpeg"],["dark_eyed_junco-26","dark_eyed_junco",7660371,6119391,"jeffreyleeisanaturalist",108394,"0f667e9af8e8d3585807da58a2f315c5dbcb7eece5f6862006fb9da68a4bef42","jpg"],["dark_eyed_junco-27","dark_eyed_junco",437957476,245430473,"eug302",106520,"394abfbcf192ecf5a76cb2f19dfa2a62070fa3874b084d1468219d6ee5db6cbf","jpeg"],["dark_eyed_junco-28","dark_eyed_junco",339439778,193239701,"rawcomposition",32258,"73fad57ed975df01c529e77eaca9eab9b00741ea2cfc91358856aefca620a10a","jpg"],["dark_eyed_junco-29","dark_eyed_junco",6198359,5055484,"glmory",108168,"d31767f42eaaf7f3527133fffb9a0271a9dbbe66fc036d5c694b605563d09965","jpg"],["house_finch-00","house_finch",117990649,72375345,"kristen163",75945,"eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f","jpeg"],["house_finch-01","house_finch",389479656,220010434,"aster-asti",82128,"a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5","jpg"],["house_finch-02","house_finch",176982307,105476125,"vicki936",22211,"c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f","jpeg"],["house_finch-03","house_finch",697940852,381438133,"ben142",196735,"a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5","jpg"],["house_finch-04","house_finch",72470599,45698380,"henrya",61724,"1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b","jpeg"],["house_finch-05","house_finch",98576538,61594129,"enspring",46784,"8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c","jpg"],["house_finch-06","house_finch",80751781,50842166,"leahmfulton",44269,"6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7","jpg"],["house_finch-07","house_finch",630196420,345777550,"truthseqr",149455,"abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c","jpg"],["house_finch-08","house_finch",214612538,126483167,"hamiltonturner",124116,"6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930","jpeg"],["house_finch-09","house_finch",213077180,125637342,"jnicat",25823,"e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8","jpeg"],["house_finch-10","house_finch",500744872,278868417,"pbaff",149626,"2684cfb1fc40d7766610a5920ead0ad0c27c338ccb4fb9ed18baddda150568b0","jpeg"],["house_finch-11","house_finch",268678834,155431721,"stevestevens",120325,"5d1e8c097c214d14ef7b895bf95769ae9bc25fa799a98bbfa6f12f2f84e6af79","jpeg"],["house_finch-12","house_finch",358373136,202884575,"kcthetc1",52329,"b9929163e4fdadef26c753437ac7af3ad550aa047b15cba131c05d4d335799fe","jpeg"],["house_finch-13","house_finch",104227663,64793290,"verdantpulsar",101003,"90fe37c477bad9ed30ab119e1a7445ffc2720e548a22bb65d66b2ff7b82337d5","jpeg"],["house_finch-14","house_finch",196156834,116218899,"kgarrett",56801,"ce03d1d70a89b5e6e0088f4307f8077573c3571b91997725ca2e6c69be80e2d0","jpeg"],["house_finch-15","house_finch",454332148,253709711,"rlaortiz",149985,"cf007ef8ac57bc0eb085c9eecf9fc95eb69a29df706998de237df24f9491c61d","jpeg"],["house_finch-16","house_finch",509969159,283798927,"damienxw",104277,"56d3656be473c362f1ccd09d15e62cbcfe9137d83bef7a3312d72444921c7805","jpg"],["house_finch-17","house_finch",665287910,295246528,"dinomariobob",74972,"5f6121c1f8dbfaccbb61c279a579c22600744eb4118c2eeef09e69c86f6e1a49","jpg"],["house_finch-18","house_finch",168402062,100871757,"k-simpkins",86922,"209a884cf7618d0b85679ae3a72f237a6d03a5a3096f6ab1b2e5503c38c50d9d","jpg"],["house_finch-19","house_finch",247557547,144258909,"aparrot1",79152,"115302ef807e6f4d132df584a20ba8644846f05dfe9fee27a7c0e07ec89d87b7","jpg"],["house_finch-20","house_finch",249874672,145517234,"vijaybarve",115757,"15538eeeb228d284f49f33d0bda77626b57fdaabe90d05f4379ffc3bbd855703","jpeg"],["house_finch-21","house_finch",379972789,214941038,"dougbrown",29723,"d7fea293bd3f92aec7be52bdfe5b904793c53cfaeacf9ffd2e762702ee10ba74","jpeg"],["house_finch-22","house_finch",250140068,145607589,"matthias55",87579,"62adc5d86cbdb613779fefa84f6f74f306b6fab5bfb86320f4ef40cd2571ff6a","jpeg"],["house_finch-23","house_finch",247757548,144364266,"nana10",123890,"10b3b6c0b4273a31597050df4218899ddcbe31cdfa60d9a46421ed0bf3d26558","jpeg"],["house_finch-24","house_finch",253927599,147557306,"kerykeion",91321,"6166a9bf59e2075a1129b344a24f3fcafccb610eb984510c5b4c3f9b3abe96af","jpeg"],["house_finch-25","house_finch",250651171,145869416,"cathartic_cathartes",55197,"1929f770ec5ca766c6b88ffdbad5fd7c27df033e12fe46109fd87dc1c835fc59","jpeg"],["house_finch-26","house_finch",244648136,142674679,"michelle_lopez",51377,"9422e42758a68c343d0487d07726819424a24c8271a94aa48c75f4c1e339be77","jpg"],["house_finch-27","house_finch",373687946,211229903,"c_dizzy",145810,"ae7289dc681f8e192886d47018ee70aa9586f08e618e57fa1fe195a5695e1779","jpeg"],["house_finch-28","house_finch",242992511,141794074,"chrisleearm",102901,"b6bd72b41f04d2b6a7855b28e2b169d5aad6663db4a0ac0d364b22ffc2512018","jpg"],["house_finch-29","house_finch",110518705,68278832,"kemper",103151,"62ee726242d1970d9bb8536e31c06635afc92b9352d1394574acb1d9d5eb26ed","jpeg"],["american_goldfinch-00","american_goldfinch",84579952,53187208,"glennberry",59673,"72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36","jpeg"],["american_goldfinch-01","american_goldfinch",12533322,9255418,"braincellsgone",55661,"6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc","jpg"],["american_goldfinch-02","american_goldfinch",131102823,80016788,"radrat",98990,"232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff","jpeg"],["american_goldfinch-03","american_goldfinch",175048222,104466897,"eug302",44231,"11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e","jpg"],["american_goldfinch-04","american_goldfinch",68849595,43390778,"mefisher",154503,"66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1","jpg"],["american_goldfinch-05","american_goldfinch",431916465,242278180,"k-simpkins",45413,"d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d","jpg"],["american_goldfinch-06","american_goldfinch",377136648,213398931,"nathan1177",66516,"7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438","jpg"],["american_goldfinch-07","american_goldfinch",230801384,135330401,"enspring",42886,"78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed","jpeg"],["american_goldfinch-08","american_goldfinch",660472044,361884286,"ben142",270599,"733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876","jpg"],["american_goldfinch-09","american_goldfinch",294667312,169935316,"dande",163470,"39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0","jpeg"],["american_goldfinch-10","american_goldfinch",143215717,86889530,"memoosborne",129438,"ef4a9a771cef9ba3c2c047eb106a6aa220236dd6aaa6aade5f4ef3a37c8abcba","jpeg"],["american_goldfinch-11","american_goldfinch",403873164,227647158,"drew_baxter",106009,"74e34c776f1b9a5d375a7dfae0e309cd42dbd133b82a1ecad4a49111ac7eed49","jpeg"],["american_goldfinch-12","american_goldfinch",481153019,267722510,"vicki936",255397,"3326ddbee3270241b681cb636466e742491f110e9841f453f5158864aadaea36","jpg"],["american_goldfinch-13","american_goldfinch",213311122,125763980,"hickl",24740,"0c295b3761bced4215519ff24a4f734773b54be98de456a4444d0819456af7dd","jpeg"],["american_goldfinch-14","american_goldfinch",417352824,234736320,"joy4birds",81109,"357014c108519543471b94f39591667d1a67d87fd1fdc4702a3a449235d6bddd","jpeg"],["american_goldfinch-15","american_goldfinch",72130720,45486482,"dctphoto",178111,"17b2f3599e20161acc17fd63bf61e9f40488b9c4d5bfaefc8cae54de42997509","jpeg"],["american_goldfinch-16","american_goldfinch",45148898,28952026,"megachile",49358,"6bbc6ca0ac074e486c20ce4c3fad5dc863cb2e908efc2b1ef97494403a351252","jpeg"],["american_goldfinch-17","american_goldfinch",133251099,81250513,"raffib128",128110,"863c76587fe1de80a84b97c2e72f38ae1cf4fa972789e006d9b0b544d696c6ef","jpeg"],["american_goldfinch-18","american_goldfinch",155797129,93957328,"nathanael15",74086,"7059ae3bdeeb48fe949e5b70b788bd28c96aecd0fe49cb7be22db9a8697bf5a5","jpg"],["american_goldfinch-19","american_goldfinch",11400438,8535447,"akneidel",26423,"4af74c1d04ddc7bbb7bb0e9eb2977a1daf48dd9b9477d71d69a7c4cb5d4f785b","jpg"],["american_goldfinch-20","american_goldfinch",145464774,88186208,"wildreturn",68204,"565d2a3b6d404e9ea0c24737ebcc5a3a82057b1452b4790df5e9552e8c50e592","jpg"],["american_goldfinch-21","american_goldfinch",55990791,35505213,"conhawn",84915,"69f768c39a2180440bdcbfc6addc5d426341e8080d4cf9ba8241d57564b3e6fb","jpg"],["american_goldfinch-22","american_goldfinch",637247336,289067166,"dinomariobob",142312,"01c63d305fce8edcc3a494f0543154c6bd6aa52368e02be84cf6c23e95b94cdd","jpg"],["american_goldfinch-23","american_goldfinch",460988055,257029007,"eric112",60314,"0db1b5f0e32d1faf861a437a20794fa18e80ea8966313933c145451c9319e2a9","jpg"],["american_goldfinch-24","american_goldfinch",59072170,37280587,"bradenjudson",23468,"827d77bf9ca9cc456e867434a07b68cdceb4a20e09eb9e6b67757c80cd31ff42","jpeg"],["american_goldfinch-25","american_goldfinch",109875643,67928198,"artemis224",217909,"99e1d7d30eb19d47c7974e9ddd7efe4d06329c952a41e15d0a4b2095b747df61","jpg"],["american_goldfinch-26","american_goldfinch",66515260,41915252,"reuvenm",58833,"90f0b687d9626fdf5d0111b794cf960cf18f3de4c8eab601fcb23c41b45c4df7","jpeg"],["american_goldfinch-27","american_goldfinch",112715850,69464521,"umamimomma",78717,"fae2edcb0901dada461a9ac6b3873d6479e8faf66bfd26a0ad47021f6bcff704","jpg"],["american_goldfinch-28","american_goldfinch",123422249,75440388,"rachel_bosley",131792,"864be3aebb7f1c8c9ed01cdf2e16b690f11354bb37e53b23d44671082afdbcfc","jpg"],["american_goldfinch-29","american_goldfinch",252964992,147051902,"carterdorscht",149077,"a8360d1774df42f034692863781af47fd030265df9bcc1fc938333fabc673c9e","jpeg"]]''')

CORPUS_NAME = "iNaturalist CC0 bird photographs (six species)"
CORPUS_RELEASE = "iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19"
CORPUS_BASE_URL = "https://inaturalist-open-data.s3.amazonaws.com/photos/"
CORPUS_LICENSE = "CC0 1.0"
CORPUS_BYTES = 19_183_071
CACHE_DIR = Path("weights/inat-birds")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CLASS_KEYS = [
    "song_sparrow",
    "chipping_sparrow",
    "white_throated_sparrow",
    "dark_eyed_junco",
    "house_finch",
    "american_goldfinch",
]
SPECIES = {
    "song_sparrow": ("Melospiza melodia", "Song Sparrow"),
    "chipping_sparrow": ("Spizella passerina", "Chipping Sparrow"),
    "white_throated_sparrow": ("Zonotrichia albicollis", "White-throated Sparrow"),
    "dark_eyed_junco": ("Junco hyemalis", "Dark-eyed Junco"),
    "house_finch": ("Haemorhous mexicanus", "House Finch"),
    "american_goldfinch": ("Spinus tristis", "American Goldfinch"),
}
CLASS_TO_ID = {name:i for i,name in enumerate(CLASS_KEYS)}

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def image_digest(image):
    rgb = image.convert("RGB")
    return sha256_bytes(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())

def fetch_dataset():
    records = []
    total_bytes = 0
    seen_pixels = set()
    for rid,label,photo_id,obs_id,observer,nbytes,digest,ext in SAMPLE_ROWS:
        path = CACHE_DIR / f"{photo_id}.{ext}"
        data = path.read_bytes() if path.is_file() else b""
        if len(data) != nbytes or sha256_bytes(data) != digest:
            url = f"{CORPUS_BASE_URL}{photo_id}/medium.{ext}"
            req = urllib.request.Request(url, headers={"User-Agent":"dimer-modern-image-workshop/1.0"})
            with urllib.request.urlopen(req, timeout=120) as response:
                data = response.read()
            if len(data) != nbytes or sha256_bytes(data) != digest:
                raise ValueError(f"{rid}: fetched bytes do not match pinned size/SHA-256")
            path.write_bytes(data)
        total_bytes += len(data)
        image = Image.open(io.BytesIO(data))
        image.load()
        image = image.convert("RGB")
        pix = image_digest(image)
        if pix in seen_pixels:
            raise ValueError(f"Duplicate decoded image detected: {rid}")
        seen_pixels.add(pix)
        records.append({
            "source_id":rid,
            "label":label,
            "label_id":CLASS_TO_ID[label],
            "image":image,
            "photo_id":photo_id,
            "observation_id":obs_id,
            "observation_url":f"https://www.inaturalist.org/observations/{obs_id}",
            "observer":observer,
            "bytes":nbytes,
            "sha256":digest,
            "pixel_sha256":pix,
            "scientific_name":SPECIES[label][0],
            "common_name":SPECIES[label][1],
        })
    if len(records) != 180:
        raise RuntimeError(f"Expected 180 images, got {len(records)}")
    if total_bytes != CORPUS_BYTES:
        raise RuntimeError(f"Expected {CORPUS_BYTES} source bytes, got {total_bytes}")
    counts = Counter(r["label"] for r in records)
    if set(counts.values()) != {30} or len(counts) != 6:
        raise RuntimeError(f"Expected 30 images/species: {counts}")
    return records

records = fetch_dataset()
print({"records":len(records),"bytes":sum(r["bytes"] for r in records),"classes":Counter(r["label"] for r in records)})

{'records': 180, 'bytes': 19183071, 'classes': Counter({'song_sparrow': 30, 'chipping_sparrow': 30, 'white_throated_sparrow': 30, 'dark_eyed_junco': 30, 'house_finch': 30, 'american_goldfinch': 30})}


## 6. Reproduce the canonical 108 / 24 / 48 split

The split matches the live ViT carrier:

- 18 train images per species;
- 4 validation images per species;
- 8 test images per species;
- seed 42.

Decoded-pixel hashes are checked for cross-split leakage.

In [4]:
def build_split(records):
    rng = random.Random(SAMPLE_SEED)
    by_label = defaultdict(list)
    for r in records:
        by_label[r["label"]].append(dict(r))
    out = {"train":[],"validation":[],"test":[]}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        out["train"].extend(pool[:18])
        out["validation"].extend(pool[18:22])
        out["test"].extend(pool[22:30])
    for name in out:
        rng.shuffle(out[name])
        out[name] = [
            {**r, "id":f"{name}-{i:03d}", "split":name}
            for i,r in enumerate(out[name])
        ]
    return out

splits = build_split(records)

seen = {}
for split_name, part in splits.items():
    for r in part:
        key = r["pixel_sha256"]
        if key in seen and seen[key] != split_name:
            raise RuntimeError(f"Image leakage across {seen[key]} and {split_name}: {r['source_id']}")
        seen[key] = split_name

expected = {"train":108,"validation":24,"test":48}
actual = {k:len(v) for k,v in splits.items()}
if actual != expected:
    raise RuntimeError(f"Split sizes {actual} != {expected}")

dataset_manifest = []
for split_name in ("train","validation","test"):
    for r in splits[split_name]:
        dataset_manifest.append({
            "id":r["id"],"source_id":r["source_id"],"split":split_name,
            "label":r["label"],"label_id":r["label_id"],"photo_id":r["photo_id"],
            "observation_id":r["observation_id"],"observer":r["observer"],
            "bytes":r["bytes"],"sha256":r["sha256"],"pixel_sha256":r["pixel_sha256"],
        })

dataset_digest = hashlib.sha256(
    json.dumps(dataset_manifest, sort_keys=True, separators=(",",":")).encode()
).hexdigest()

print({"split_sizes":actual,"dataset_digest":dataset_digest})

{'split_sizes': {'train': 108, 'validation': 24, 'test': 48}, 'dataset_digest': '842433b772bbc8649d059c2def5415c150c396fa953464e47331a5a6bfa3b96a'}


## 7. Six immutable model snapshots

Each model is staged from its exact Hugging Face revision and every manifest-listed file is verified by byte size and SHA-256 before loading.

The notebook loads weights from the verified local `model.safetensors`; no mutable `main` fallback is permitted.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [5]:
MANIFESTS = json.loads(r'''{"resnet50":{"format":"dimer_hf_snapshot","formatVersion":1,"modelKey":"resnet50-a1","modelId":"timm/resnet50.a1_in1k","revision":"767268603ca0cb0bfe326fa87277f19c419566ef","files":[{"path":"README.md","bytes":38435,"sha256":"1bf90a24888cff1da80e2e7b2e341fad1e4e448c19d92eb5bc08468393251721"},{"path":"config.json","bytes":756,"sha256":"ff00d936f02c3b5f9f80a169d651ed28f9b6dc9cb1fd0464f92da4761caaac50"},{"path":"model.safetensors","bytes":102469840,"sha256":"773525d5821de224f8f30c33377b7a795d7863e08522698200d3217d3f2a41bb"}],"totalBytes":102509031},"mobilenetv4":{"format":"dimer_hf_snapshot","formatVersion":1,"modelKey":"mobilenetv4-conv-small","modelId":"timm/mobilenetv4_conv_small.e2400_r224_in1k","revision":"331fb803779522b685cf942e15f914fb6741c1eb","files":[{"path":"README.md","bytes":13066,"sha256":"864651a7a9c0457023728da202cb1119abf3fa0872ded374ba736efde6454eb7"},{"path":"config.json","bytes":681,"sha256":"ecd20bcf1287aaa88129a736d188904d0b134c96e152b55b241be895657fe41a"},{"path":"model.safetensors","bytes":15223016,"sha256":"7a7102ec18f62bbfb555b6fe829bbb5af749516b84174926c29ffdfdfc03aec4"}],"totalBytes":15236763},"convnext":{"format":"dimer_hf_snapshot","formatVersion":1,"modelKey":"convnext-tiny-in12k","modelId":"timm/convnext_tiny.in12k_ft_in1k","revision":"aa096f03029c7f0ec052013f64c819b34f8ad790","files":[{"path":"README.md","bytes":16006,"sha256":"8f702c1663d3ccade5f47673d8924cc76f86880129b12a2958a9d094a8ec15cc"},{"path":"config.json","bytes":662,"sha256":"7333fa40fddb5a875bcf1bdcfe30916c925c27c6155834b900e2c4cc08dc4333"},{"path":"model.safetensors","bytes":114374272,"sha256":"a1aefa409b513cf209b085424eb3efffe4e4a9f511491bc2c12ea35209e6bb95"}],"totalBytes":114390940},"vit":{"format":"dimer_hf_snapshot","formatVersion":1,"modelKey":"vit-base-p16-224","modelId":"timm/vit_base_patch16_224.orig_in21k_ft_in1k","revision":"e0bd370de6799e8d1f47a911174ff4c3708e2323","files":[{"path":"README.md","bytes":3409,"sha256":"c5171b9aefd75755fac887b960b4d1df8cfa1f90568fc03ff5a5d5b74566664d"},{"path":"config.json","bytes":585,"sha256":"91aa54e1244ba735215d4ec117b62c1a00a9f4a744b0e0543b16b901eaf73784"},{"path":"model.safetensors","bytes":346284714,"sha256":"669b949ea91fd19217f200cee259780bde32210c1eb9a5af3859f0dd8346b2ec"}],"totalBytes":346288708},"swinv2":{"format":"dimer_hf_snapshot","formatVersion":1,"modelKey":"swinv2-tiny-window8-256-ms-in1k","modelId":"timm/swinv2_tiny_window8_256.ms_in1k","revision":"650d02aabf05e8adbd060a739ab39e39f53da639","files":[{"path":"config.json","bytes":632,"sha256":"75b22a3513f40757680f3f3b204adee1aa751f07eb630ee46200ed2c8a22c767"},{"path":"model.safetensors","bytes":114918618,"sha256":"c47f52b4556ff4436aa9502f5efbc93aac77fdab42d9d70dd845931757ff5d65"}],"totalBytes":114919250},"eva02":{"format":"dimer_hf_snapshot","formatVersion":1,"modelKey":"eva02-base-448","modelId":"timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k","revision":"81063ecfe9c381a16a19d06f396d6c7011aa426a","files":[{"path":"README.md","bytes":5454,"sha256":"b347298844860ff288638f1d7c0baece71d9fe1cd796eefbfbf3f0780175ba07"},{"path":"config.json","bytes":653,"sha256":"d62e73f578eb363032000b6bfcc57764c6a5d10ee2562cbfe3f105d795e1c4be"},{"path":"model.safetensors","bytes":348492484,"sha256":"533937d6f9f8f8d4f50627ef0d00829a5015861a5b67e1c69c5a5e45b7dc2609"}],"totalBytes":348498591}}''')

from huggingface_hub import hf_hub_download

MODEL_REGISTRY = {
    "resnet50":{
        "display":"ResNet-50",
        "timm_name":"resnet50.a1_in1k",
        "family":"Residual CNN","license":"Apache-2.0","declared_input":224,
    },
    "mobilenetv4":{
        "display":"MobileNetV4-Conv-Small",
        "timm_name":"mobilenetv4_conv_small.e2400_r224_in1k",
        "family":"Efficient/mobile CNN","license":"Apache-2.0","declared_input":224,
    },
    "convnext":{
        "display":"ConvNeXt-Tiny",
        "timm_name":"convnext_tiny.in12k_ft_in1k",
        "family":"Modern ConvNet","license":"Apache-2.0","declared_input":224,
    },
    "vit":{
        "display":"ViT-B/16",
        "timm_name":"vit_base_patch16_224.orig_in21k_ft_in1k",
        "family":"Global Vision Transformer","license":"Apache-2.0","declared_input":224,
    },
    "swinv2":{
        "display":"SwinV2-Tiny",
        "timm_name":"swinv2_tiny_window8_256.ms_in1k",
        "family":"Hierarchical windowed transformer","license":"MIT","declared_input":256,
    },
    "eva02":{
        "display":"EVA-02 Base 448",
        "timm_name":"eva02_base_patch14_448.mim_in22k_ft_in22k_in1k",
        "family":"Modern vision transformer","license":"MIT","declared_input":448,
    },
}
for key in MODEL_REGISTRY:
    MODEL_REGISTRY[key]["manifest"] = MANIFESTS[key]

SNAPSHOT_ROOT = Path("weights/model-snapshots")
SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def stage_snapshot(key):
    spec = MODEL_REGISTRY[key]
    manifest = spec["manifest"]
    root = SNAPSHOT_ROOT / manifest["modelKey"]
    root.mkdir(parents=True, exist_ok=True)
    (root/"dimer-base-manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    t0 = time.perf_counter()
    for entry in manifest["files"]:
        path = root / entry["path"]
        if not path.is_file():
            path.parent.mkdir(parents=True, exist_ok=True)
            hf_hub_download(
                repo_id=manifest["modelId"],
                filename=entry["path"],
                revision=manifest["revision"],
                local_dir=str(root),
            )
    for entry in manifest["files"]:
        path = root / entry["path"]
        if path.stat().st_size != entry["bytes"]:
            raise RuntimeError(f"{key} {entry['path']}: size mismatch")
        if sha256_file(path) != entry["sha256"]:
            raise RuntimeError(f"{key} {entry['path']}: SHA-256 mismatch")
    return root, time.perf_counter()-t0

print("Registered models:")
for key,spec in MODEL_REGISTRY.items():
    print(f"  {key:<12} {spec['display']:<27} {spec['manifest']['modelId']} @ {spec['manifest']['revision'][:12]}…")

Registered models:
  resnet50     ResNet-50                   timm/resnet50.a1_in1k @ 767268603ca0…
  mobilenetv4  MobileNetV4-Conv-Small      timm/mobilenetv4_conv_small.e2400_r224_in1k @ 331fb8037795…
  convnext     ConvNeXt-Tiny               timm/convnext_tiny.in12k_ft_in1k @ aa096f03029c…
  vit          ViT-B/16                    timm/vit_base_patch16_224.orig_in21k_ft_in1k @ e0bd370de679…
  swinv2       SwinV2-Tiny                 timm/swinv2_tiny_window8_256.ms_in1k @ 650d02aabf05…
  eva02        EVA-02 Base 448             timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k @ 81063ecfe9c3…


## 8. Common representation and probe methodology

For every checkpoint:

1. load the exact pretrained architecture;
2. freeze every backbone parameter;
3. apply its own native `timm` evaluation preprocessing;
4. call `forward_features` then `forward_head(..., pre_logits=True)`;
5. cache one pooled representation per image;
6. fit feature mean/std **on training features only**;
7. evaluate a 5-NN baseline;
8. train a zero-initialized `nn.Linear(feature_dim, 6)` probe;
9. select the probe epoch using validation cross-entropy;
10. evaluate test once;
11. export the probe as SafeTensors and prove fresh reload parity.

In [6]:
def confusion_matrix_np(y_true,y_pred,n_classes=6):
    cm = np.zeros((n_classes,n_classes),dtype=int)
    for t,p in zip(y_true,y_pred,strict=True):
        cm[int(t),int(p)] += 1
    return cm

def class_metrics(cm):
    rows=[]
    for i in range(cm.shape[0]):
        tp=cm[i,i]; fp=cm[:,i].sum()-tp; fn=cm[i,:].sum()-tp
        support=cm[i,:].sum()
        precision=tp/(tp+fp) if tp+fp else 0.0
        recall=tp/(tp+fn) if tp+fn else 0.0
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
        rows.append({"class_id":i,"class_label":CLASS_KEYS[i],"support":int(support),
                     "precision":float(precision),"recall":float(recall),"f1":float(f1)})
    return rows

def classification_metrics(y_true, probs):
    y_true=np.asarray(y_true,dtype=int)
    probs=np.asarray(probs,dtype=float)
    pred=probs.argmax(axis=1)
    cm=confusion_matrix_np(y_true,pred,len(CLASS_KEYS))
    per=class_metrics(cm)
    top3=np.argsort(-probs,axis=1)[:,:3]
    log_loss=float(-np.mean(np.log(np.maximum(probs[np.arange(len(y_true)),y_true],1e-12))))
    return {
        "accuracy":float(np.mean(pred==y_true)),
        "macro_f1":float(np.mean([r["f1"] for r in per])),
        "top3_accuracy":float(np.mean([y_true[i] in top3[i] for i in range(len(y_true))])),
        "log_loss":log_loss,
        "confusion_matrix":cm,
        "per_class":per,
        "predicted_ids":pred,
    }

def standardize_from_train(train_x,*others):
    mean=train_x.mean(axis=0,keepdims=True).astype(np.float32)
    std=train_x.std(axis=0,keepdims=True).astype(np.float32)
    std=np.maximum(std,1e-6)
    return mean,std,[(x-mean)/std for x in (train_x,*others)]

def knn_predict(train_x,train_y,test_x,k=5):
    train_n=train_x/np.maximum(np.linalg.norm(train_x,axis=1,keepdims=True),1e-12)
    test_n=test_x/np.maximum(np.linalg.norm(test_x,axis=1,keepdims=True),1e-12)
    sims=test_n@train_n.T
    predictions=[]
    probs=[]
    for row in sims:
        idx=np.argsort(-row,kind="stable")[:k]
        vote_counts=np.zeros(len(CLASS_KEYS),dtype=int)
        sim_sums=np.zeros(len(CLASS_KEYS),dtype=float)
        for j in idx:
            c=int(train_y[j]); vote_counts[c]+=1; sim_sums[c]+=float(row[j])
        order=sorted(range(len(CLASS_KEYS)), key=lambda c:(-vote_counts[c],-sim_sums[c],c))
        predictions.append(order[0])
        p=vote_counts.astype(float)
        p=p/p.sum()
        probs.append(p)
    return np.array(predictions),np.array(probs,dtype=np.float32)

def fit_probe(train_x,train_y,val_x,val_y,test_x,
              epochs=PROBE_EPOCHS,lr=PROBE_LR,weight_decay=PROBE_WEIGHT_DECAY):
    mean,std,(tr,va,te)=standardize_from_train(train_x,val_x,test_x)
    xtr=torch.from_numpy(tr.astype(np.float32))
    ytr=torch.from_numpy(np.asarray(train_y,dtype=np.int64))
    xva=torch.from_numpy(va.astype(np.float32))
    yva=torch.from_numpy(np.asarray(val_y,dtype=np.int64))
    xte=torch.from_numpy(te.astype(np.float32))

    layer=torch.nn.Linear(tr.shape[1],len(CLASS_KEYS))
    torch.nn.init.zeros_(layer.weight); torch.nn.init.zeros_(layer.bias)
    opt=torch.optim.AdamW(layer.parameters(),lr=lr,weight_decay=weight_decay)

    best_loss=float("inf"); best_epoch=None; best_state=None
    start=time.perf_counter()
    for epoch in range(1,epochs+1):
        layer.train()
        opt.zero_grad(set_to_none=True)
        loss=F.cross_entropy(layer(xtr),ytr)
        loss.backward()
        opt.step()
        layer.eval()
        with torch.inference_mode():
            val_loss=float(F.cross_entropy(layer(xva),yva))
        if val_loss < best_loss - 1e-12:
            best_loss=val_loss; best_epoch=epoch
            best_state={k:v.detach().clone() for k,v in layer.state_dict().items()}
    training_seconds=time.perf_counter()-start
    layer.load_state_dict(best_state)
    layer.eval()
    with torch.inference_mode():
        test_probs=torch.softmax(layer(xte),dim=1).numpy().astype(np.float32)
        val_probs=torch.softmax(layer(xva),dim=1).numpy().astype(np.float32)
    return {
        "layer":layer,"mean":mean.squeeze(0),"std":std.squeeze(0),
        "best_epoch":best_epoch,"validation_loss":best_loss,
        # Honesty flags: a first-epoch pick means the probe barely trained; a cap pick means validation loss was
        # still falling (common when the 24 validation images are already separable).
        "selected_at_first_epoch":best_epoch==1,"selected_at_epoch_cap":best_epoch==epochs,
        "test_probs":test_probs,"val_probs":val_probs,
        "training_seconds":training_seconds,
    }

def representation_geometry(test_x,test_y):
    x=test_x/np.maximum(np.linalg.norm(test_x,axis=1,keepdims=True),1e-12)
    sim=x@x.T
    same=[]; diff=[]
    for i in range(len(x)):
        for j in range(i+1,len(x)):
            (same if test_y[i]==test_y[j] else diff).append(float(sim[i,j]))
    return {
        "mean_same_class_cosine":float(np.mean(same)),
        "mean_different_class_cosine":float(np.mean(diff)),
        "separability_gap":float(np.mean(same)-np.mean(diff)),
    }

## 9. Majority baseline

The dataset is exactly balanced across six species, so a deterministic majority-class classifier has expected accuracy `1/6 ≈ 16.67%`.

> **Before you run it:** predict whether this simple reference will be easy or difficult for the learned model(s) to beat. Record the baseline before interpreting the more complex result.

In [7]:
majority_accuracy=1.0/len(CLASS_KEYS)
print({"majority_accuracy":majority_accuracy})

{'majority_accuracy': 0.16666666666666666}


## 10. Sequential backbone workflow

The next cell executes the same procedure for all six checkpoints.

No backbone gradients are enabled. Model-specific differences in preprocessing are preserved, including 224×224, 256×256 and 448×448 native inputs.

In [8]:
from timm.data import resolve_model_data_config, create_transform

OUT_ROOT=Path(OUTPUT_DIR)
(OUT_ROOT/"features").mkdir(parents=True,exist_ok=True)
(OUT_ROOT/"probes").mkdir(parents=True,exist_ok=True)
(OUT_ROOT/"pca").mkdir(parents=True,exist_ok=True)

def load_backbone(key):
    spec=MODEL_REGISTRY[key]
    root,verify_seconds=stage_snapshot(key)
    t0=time.perf_counter()
    model=timm.create_model(spec["timm_name"],pretrained=False)
    state=load_file(str(root/"model.safetensors"),device="cpu")
    # Some checkpoints (SwinV2) carry buffers that timm now rebuilds at construction and does not persist
    # (attention masks). Drop such an entry only when it is bit-identical to the rebuilt buffer; everything
    # else still goes through a strict load.
    persistent=set(model.state_dict())
    rebuilt={n:b for n,b in model.named_buffers() if n not in persistent}
    for name in [k for k in state if k in rebuilt]:
        if not torch.equal(state[name].to(rebuilt[name].dtype),rebuilt[name].cpu()):
            raise RuntimeError(f"{key}: checkpoint buffer {name} differs from the one timm rebuilds")
        del state[name]
    result=model.load_state_dict(state,strict=True)
    if result.missing_keys or result.unexpected_keys:
        raise RuntimeError(f"{key}: state mismatch {result}")
    del state
    model=model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    if sum(p.requires_grad for p in model.parameters()) != 0:
        raise RuntimeError(f"{key}: backbone is not fully frozen")
    data_config=resolve_model_data_config(model)
    transform=create_transform(**data_config,is_training=False)
    input_size=int(data_config["input_size"][-1])
    if input_size != spec["declared_input"]:
        raise RuntimeError(f"{key}: resolved input {input_size} != declared {spec['declared_input']}")
    return model,transform,data_config,root,verify_seconds,time.perf_counter()-t0

def extract_features(model,transform,part):
    feats=[]; ids=[]; labels=[]; source_ids=[]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0=time.perf_counter()
    for start in range(0,len(part),FEATURE_BATCH_SIZE):
        batch=part[start:start+FEATURE_BATCH_SIZE]
        tensor=torch.stack([transform(r["image"]) for r in batch]).to(DEVICE)
        with torch.inference_mode():
            f=model.forward_features(tensor)
            z=model.forward_head(f,pre_logits=True)
        if z.ndim != 2 or z.shape[0] != len(batch):
            raise RuntimeError(f"Unexpected pre-logit shape {tuple(z.shape)}")
        if not torch.isfinite(z).all():
            raise RuntimeError("Non-finite pre-logit features")
        feats.append(z.float().cpu().numpy())
        ids.extend(r["id"] for r in batch)
        source_ids.extend(r["source_id"] for r in batch)
        labels.extend(r["label_id"] for r in batch)
    elapsed=time.perf_counter()-t0
    peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
    return {
        "features":np.vstack(feats).astype(np.float32),
        "ids":np.array(ids,dtype=str),
        "source_ids":np.array(source_ids,dtype=str),
        "labels":np.array(labels,dtype=np.int64),
        "seconds":elapsed,"peak_gpu_memory_bytes":peak,
    }

def latency_measure(model,transform,image):
    x=transform(image).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        for _ in range(2):
            _=model.forward_head(model.forward_features(x),pre_logits=True)
    values=[]
    for _ in range(LATENCY_REPEATS):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0=time.perf_counter()
        with torch.inference_mode():
            _=model.forward_head(model.forward_features(x),pre_logits=True)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        values.append(time.perf_counter()-t0)
    return float(np.median(values)),values

def save_probe_artifact(key,spec,probe,feature_dim,train_digest,val_digest):
    root=OUT_ROOT/"probes"/key
    root.mkdir(parents=True,exist_ok=True)
    path=root/"probe.safetensors"
    tensors={
        "classifier.weight":probe["layer"].weight.detach().cpu().contiguous(),
        "classifier.bias":probe["layer"].bias.detach().cpu().contiguous(),
        "feature_mean":torch.from_numpy(probe["mean"]).contiguous(),
        "feature_std":torch.from_numpy(probe["std"]).contiguous(),
    }
    save_file(tensors,str(path))
    base_sha=next(x["sha256"] for x in spec["manifest"]["files"] if x["path"]=="model.safetensors")
    manifest={
        "format":"dimer_linear_probe",
        "format_version":1,
        "base_model_id":spec["manifest"]["modelId"],
        "base_model_revision":spec["manifest"]["revision"],
        "base_weight_sha256":base_sha,
        "feature_interface":"pre_logits",
        "feature_dim":feature_dim,
        "class_order":CLASS_KEYS,
        "probe":{"best_epoch":probe["best_epoch"],"optimizer":"AdamW","learning_rate":PROBE_LR,
                 "weight_decay":PROBE_WEIGHT_DECAY},
        "training_sample_digest":train_digest,
        "validation_sample_digest":val_digest,
        "probe_file":{"bytes":path.stat().st_size,"sha256":sha256_file(path)},
    }
    (root/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    return path,manifest

def reload_probe_verify(path,manifest,spec,test_x,reference_probs):
    base_sha=next(x["sha256"] for x in spec["manifest"]["files"] if x["path"]=="model.safetensors")
    if manifest["base_model_id"] != spec["manifest"]["modelId"] or manifest["base_model_revision"] != spec["manifest"]["revision"]:
        raise RuntimeError("Probe/base model identity mismatch")
    if manifest["base_weight_sha256"] != base_sha:
        raise RuntimeError("Probe/base weight digest mismatch")
    state=load_file(str(path),device="cpu")
    dim=int(manifest["feature_dim"])
    if test_x.shape[1] != dim:
        raise RuntimeError("Probe feature dimension mismatch")
    fresh=torch.nn.Linear(dim,len(CLASS_KEYS))
    fresh.load_state_dict({"weight":state["classifier.weight"],"bias":state["classifier.bias"]})
    mean=state["feature_mean"].numpy()
    std=state["feature_std"].numpy()
    x=torch.from_numpy(((test_x-mean)/std).astype(np.float32))
    fresh.eval()
    with torch.inference_mode():
        probs=torch.softmax(fresh(x),dim=1).numpy()
    diff=float(np.max(np.abs(probs-reference_probs)))
    if diff > 1e-6 or not np.array_equal(probs.argmax(1),reference_probs.argmax(1)):
        raise RuntimeError(f"Probe reload parity failed: max diff {diff}")
    return diff

def split_digest(part):
    payload=[[r["source_id"],r["label_id"],r["pixel_sha256"]] for r in part]
    return hashlib.sha256(json.dumps(payload,separators=(",",":")).encode()).hexdigest()

ALL_RESULTS={}
ALL_FEATURES={}

fixed_latency_image=splits["test"][0]["image"]

for key,spec in MODEL_REGISTRY.items():
    print(f"\n=== {spec['display']} ===")
    model,transform,data_cfg,snapshot_dir,verify_s,load_s=load_backbone(key)
    parameter_count=sum(p.numel() for p in model.parameters())
    # Pre-logit width: timm's head_hidden_size (MobileNetV4 adds a 960->1280 head conv, so num_features is not it).
    feature_dim=int(getattr(model,"head_hidden_size",model.num_features))

    median_latency,latency_values=latency_measure(model,transform,fixed_latency_image)

    tr=extract_features(model,transform,splits["train"])
    va=extract_features(model,transform,splits["validation"])
    te=extract_features(model,transform,splits["test"])

    if tr["features"].shape[1] != feature_dim or va["features"].shape[1] != feature_dim or te["features"].shape[1] != feature_dim:
        raise RuntimeError(f"{key}: feature dimension does not match the pre-logit width {feature_dim}")

    mean,std,(tr_std,va_std,te_std)=standardize_from_train(tr["features"],va["features"],te["features"])
    knn_pred,knn_probs=knn_predict(tr_std,tr["labels"],te_std,KNN_K)
    knn_metrics=classification_metrics(te["labels"],knn_probs)

    probe=fit_probe(tr["features"],tr["labels"],va["features"],va["labels"],te["features"])
    probe_metrics=classification_metrics(te["labels"],probe["test_probs"])

    geom=representation_geometry(te["features"],te["labels"])

    feature_path=OUT_ROOT/"features"/f"{key}.npz"
    np.savez_compressed(
        feature_path,
        train_ids=tr["ids"],train_source_ids=tr["source_ids"],train_labels=tr["labels"],train_features=tr["features"],
        validation_ids=va["ids"],validation_source_ids=va["source_ids"],validation_labels=va["labels"],validation_features=va["features"],
        test_ids=te["ids"],test_source_ids=te["source_ids"],test_labels=te["labels"],test_features=te["features"],
    )

    probe_path,probe_manifest=save_probe_artifact(
        key,spec,probe,feature_dim,split_digest(splits["train"]),split_digest(splits["validation"])
    )
    reload_diff=reload_probe_verify(probe_path,probe_manifest,spec,te["features"],probe["test_probs"])

    input_size=int(data_cfg["input_size"][-1])
    peak=max(x for x in [tr["peak_gpu_memory_bytes"],va["peak_gpu_memory_bytes"],te["peak_gpu_memory_bytes"]] if x is not None) if torch.cuda.is_available() else None

    ALL_FEATURES[key]={"train":tr,"validation":va,"test":te}
    ALL_RESULTS[key]={
        "model":spec["display"],"family":spec["family"],"feature_dim":feature_dim,
        "parameter_count":parameter_count,"weight_bytes":next(x["bytes"] for x in spec["manifest"]["files"] if x["path"]=="model.safetensors"),
        "native_input_size":input_size,"snapshot_verify_seconds":verify_s,"model_load_seconds":load_s,
        "median_single_image_latency_s":median_latency,"latency_values":latency_values,
        "feature_extraction_seconds":tr["seconds"]+va["seconds"]+te["seconds"],
        "probe_training_seconds":probe["training_seconds"],"peak_gpu_memory_bytes":peak,
        "knn_metrics":knn_metrics,"probe_metrics":probe_metrics,
        "best_epoch":probe["best_epoch"],"validation_log_loss":probe["validation_loss"],
        "selected_at_first_epoch":probe["selected_at_first_epoch"],"selected_at_epoch_cap":probe["selected_at_epoch_cap"],
        "geometry":geom,"probe_reload_max_abs_probability_diff":reload_diff,
        "test_probs":probe["test_probs"],"test_pred":probe_metrics["predicted_ids"],
        "probe_manifest":probe_manifest,
    }

    print({
        "feature_dim":feature_dim,
        "5nn_accuracy":knn_metrics["accuracy"],
        "probe_accuracy":probe_metrics["accuracy"],
        "macro_f1":probe_metrics["macro_f1"],
        "best_epoch":probe["best_epoch"],
        "latency_s":median_latency,
    })

    del model,probe
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


=== ResNet-50 ===


README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

{'feature_dim': 2048, '5nn_accuracy': 0.5208333333333334, 'probe_accuracy': 0.5, 'macro_f1': 0.4668915879442195, 'best_epoch': 35, 'latency_s': 0.008366504500031624}

=== MobileNetV4-Conv-Small ===


...

{'feature_dim': 768, '5nn_accuracy': 0.6458333333333334, 'probe_accuracy': 0.6666666666666666, 'macro_f1': 0.6435947265668629, 'best_epoch': 34, 'latency_s': 0.015162886999974035}

=== EVA-02 Base 448 ===


README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/348M [00:00<?, ?B/s]

{'feature_dim': 768, '5nn_accuracy': 0.8333333333333334, 'probe_accuracy': 0.8958333333333334, 'macro_f1': 0.8957080200501254, 'best_epoch': 1000, 'latency_s': 0.07853578900005687}


## 11. Main comparison

The table below compares representation-only 5-NN performance with the learned linear probe.

No automatic overall winner is declared.

> **What to notice.** Compare the learned-model result with the baseline/reference first, then use the secondary diagnostics to explain the behavior. Do not infer a universal model ranking from one tutorial sample and configuration.

In [9]:
print(f"{'Model':<27} {'Feat':>6} {'5NN Acc':>9} {'Probe Acc':>10} {'Macro F1':>9} {'Top-3':>8} {'LogLoss':>8}")
print("-"*90)
for key,spec in MODEL_REGISTRY.items():
    r=ALL_RESULTS[key]
    k=r["knn_metrics"]; p=r["probe_metrics"]
    print(
        f"{r['model']:<27} {r['feature_dim']:>6} {k['accuracy']:>9.3f} "
        f"{p['accuracy']:>10.3f} {p['macro_f1']:>9.3f} {p['top3_accuracy']:>8.3f} {p['log_loss']:>8.3f}"
    )

Model                         Feat   5NN Acc  Probe Acc  Macro F1    Top-3  LogLoss
------------------------------------------------------------------------------------------
ResNet-50                     2048     0.521      0.500     0.467    0.854    1.754
MobileNetV4-Conv-Small        1280     0.396      0.500     0.505    0.812    1.448
ConvNeXt-Tiny                  768     0.771      0.729     0.727    0.917    1.133
ViT-B/16                       768     0.812      0.792     0.789    0.958    0.897
SwinV2-Tiny                    768     0.646      0.667     0.644    0.875    1.055
EVA-02 Base 448                768     0.833      0.896     0.896    0.979    0.367


## 12. Prediction exercise

Before interpreting resource tradeoffs, consider:

1. Which checkpoint produced the strongest frozen representation?
2. Did the largest checkpoint also produce the highest probe accuracy?
3. Did the highest-resolution EVA-02 representation justify its compute cost?
4. Which model gave the best result under a small weight/latency envelope?
5. Did 5-NN and the learned linear probe rank the representations similarly?

Use the measured table rather than assumptions about architecture families.

In [10]:
for key,r in ALL_RESULTS.items():
    print({
        "model":r["model"],
        "probe_accuracy":r["probe_metrics"]["accuracy"],
        "weight_mb":r["weight_bytes"]/1e6,
        "median_latency_ms":r["median_single_image_latency_s"]*1000,
        "input_size":r["native_input_size"],
    })

{'model': 'ResNet-50', 'probe_accuracy': 0.5, 'weight_mb': 102.46984, 'median_latency_ms': 8.366504500031624, 'input_size': 224}
{'model': 'MobileNetV4-Conv-Small', 'probe_accuracy': 0.5, 'weight_mb': 15.223016, 'median_latency_ms': 5.073190500070268, 'input_size': 224}
{'model': 'ConvNeXt-Tiny', 'probe_accuracy': 0.7291666666666666, 'weight_mb': 114.374272, 'median_latency_ms': 9.401498999977775, 'input_size': 224}
{'model': 'ViT-B/16', 'probe_accuracy': 0.7916666666666666, 'weight_mb': 346.284714, 'median_latency_ms': 22.027933499998653, 'input_size': 224}
{'model': 'SwinV2-Tiny', 'probe_accuracy': 0.6666666666666666, 'weight_mb': 114.918618, 'median_latency_ms': 15.162886999974035, 'input_size': 256}
{'model': 'EVA-02 Base 448', 'probe_accuracy': 0.8958333333333334, 'weight_mb': 348.492484, 'median_latency_ms': 78.53578900005687, 'input_size': 448}


## 13. Per-species confusion and cross-model consensus

A single aggregate accuracy can hide species-specific behavior.

For every held-out photo the notebook records how many of the six probes predict the gold species, and whether the case is unanimous, majority-correct, split, or a shared hard case.

In [11]:
test_records=splits["test"]
prediction_rows=[]
for key,r in ALL_RESULTS.items():
    probs=r["test_probs"]
    pred=probs.argmax(1)
    for i,record in enumerate(test_records):
        order=np.argsort(-probs[i])
        prediction_rows.append({
            "image_id":record["id"],"source_id":record["source_id"],"gold_label":record["label"],
            "model":r["model"],"predicted_label":CLASS_KEYS[int(order[0])],
            "correct":bool(int(order[0])==record["label_id"]),
            "top1_score":float(probs[i,order[0]]),
            "top2_label":CLASS_KEYS[int(order[1])],"top2_score":float(probs[i,order[1]]),
            "margin":float(probs[i,order[0]]-probs[i,order[1]]),
        })

consensus_rows=[]
for i,record in enumerate(test_records):
    preds=[int(ALL_RESULTS[k]["test_pred"][i]) for k in MODEL_REGISTRY]
    correct=sum(p==record["label_id"] for p in preds)
    unique=len(set(preds))
    if correct==6: category="unanimous_correct"
    elif correct>=4: category="majority_correct"
    elif correct>=2: category="split"
    else: category="shared_hard_case"
    consensus_rows.append({
        "image_id":record["id"],"source_id":record["source_id"],"gold_label":record["label"],
        "models_correct":correct,"unique_predictions":unique,"difficulty_category":category,
    })

print(Counter(r["difficulty_category"] for r in consensus_rows))

Counter({'majority_correct': 19, 'unanimous_correct': 14, 'split': 8, 'shared_hard_case': 7})


## 14. Representation geometry

For each backbone, held-out raw pre-logit vectors are L2-normalized and compared by cosine similarity.

`separability_gap = mean same-class cosine − mean different-class cosine`

This is a diagnostic representation statistic, not an accuracy measure.

In [12]:
for key,r in ALL_RESULTS.items():
    print(r["model"], r["geometry"])

ResNet-50 {'mean_same_class_cosine': 0.45765095283942564, 'mean_different_class_cosine': 0.3754467140262326, 'separability_gap': 0.08220423881319305}
MobileNetV4-Conv-Small {'mean_same_class_cosine': 0.375933157634877, 'mean_different_class_cosine': 0.3240144483589878, 'separability_gap': 0.05191870927588921}
ConvNeXt-Tiny {'mean_same_class_cosine': 0.4830578743435797, 'mean_different_class_cosine': 0.2565958099226312, 'separability_gap': 0.22646206442094852}
ViT-B/16 {'mean_same_class_cosine': 0.4154263631263304, 'mean_different_class_cosine': 0.2618406839674814, 'separability_gap': 0.15358567915884902}
SwinV2-Tiny {'mean_same_class_cosine': 0.41618874365286457, 'mean_different_class_cosine': 0.24465607220724148, 'separability_gap': 0.1715326714456231}
EVA-02 Base 448 {'mean_same_class_cosine': 0.5205427831083181, 'mean_different_class_cosine': 0.21842544105359896, 'separability_gap': 0.3021173420547192}


## 15. Data-efficiency experiment

Because features are already cached, additional probes are cheap.

Nested training subsets use exactly the same images across all models:

- 6 images/species = 36 train images
- 12 images/species = 72
- 18 images/species = 108

Validation and test remain unchanged.

In [13]:
data_efficiency_rows=[]

def nested_train_indices(part,per_class):
    selected=[]
    for class_id in range(len(CLASS_KEYS)):
        candidates=[
            (r["source_id"],i)
            for i,r in enumerate(part)
            if r["label_id"]==class_id
        ]
        candidates.sort()
        selected.extend(i for _sid,i in candidates[:per_class])
    return sorted(selected)

if RUN_DATA_EFFICIENCY:
    for per_class in DATA_EFFICIENCY_PER_CLASS:
        idx=nested_train_indices(splits["train"],per_class)
        for key,r in ALL_RESULTS.items():
            feat=ALL_FEATURES[key]
            probe=fit_probe(
                feat["train"]["features"][idx],feat["train"]["labels"][idx],
                feat["validation"]["features"],feat["validation"]["labels"],
                feat["test"]["features"],
            )
            m=classification_metrics(feat["test"]["labels"],probe["test_probs"])
            data_efficiency_rows.append({
                "model":r["model"],"model_key":key,
                "train_images_per_class":per_class,"train_images":len(idx),
                "best_epoch":probe["best_epoch"],"validation_loss":probe["validation_loss"],
                "test_accuracy":m["accuracy"],"test_macro_f1":m["macro_f1"],
            })
    for key in MODEL_REGISTRY:
        rows=[x for x in data_efficiency_rows if x["model_key"]==key]
        print(ALL_RESULTS[key]["model"], [(x["train_images_per_class"],round(x["test_accuracy"],3)) for x in rows])
else:
    print("Data-efficiency experiment disabled.")

ResNet-50 [(6, 0.5), (12, 0.583), (18, 0.5)]
MobileNetV4-Conv-Small [(6, 0.458), (12, 0.521), (18, 0.5)]
ConvNeXt-Tiny [(6, 0.771), (12, 0.771), (18, 0.729)]
ViT-B/16 [(6, 0.771), (12, 0.812), (18, 0.792)]
SwinV2-Tiny [(6, 0.542), (12, 0.688), (18, 0.667)]
EVA-02 Base 448 [(6, 0.896), (12, 0.917), (18, 0.896)]


## 16. Train-fitted PCA views

For visualization only, two-component PCA is fitted on each backbone's standardized **training** features, then applied to its test features.

The projections do not influence model selection.

In [14]:
pca_rows=[]
if RUN_PCA_VISUALIZATION:
    for key,r in ALL_RESULTS.items():
        feat=ALL_FEATURES[key]
        mean,std,(tr,te)=standardize_from_train(feat["train"]["features"],feat["test"]["features"])
        tr_center=tr-tr.mean(axis=0,keepdims=True)
        _u,_s,vt=np.linalg.svd(tr_center,full_matrices=False)
        components=vt[:2].T
        te2=(te-tr.mean(axis=0,keepdims=True))@components

        fig=plt.figure(figsize=(7,5))
        ax=fig.add_subplot(111)
        for class_id,label in enumerate(CLASS_KEYS):
            mask=feat["test"]["labels"]==class_id
            ax.scatter(te2[mask,0],te2[mask,1],label=SPECIES[label][1],alpha=0.8)
            for point,source_id in zip(te2[mask],feat["test"]["source_ids"][mask],strict=True):
                pca_rows.append({"model":r["model"],"model_key":key,"source_id":str(source_id),
                                 "class_id":class_id,"class_label":label,
                                 "pc1":float(point[0]),"pc2":float(point[1])})
        ax.set_title(f"{r['model']} — train-fitted PCA of test features")
        ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(OUT_ROOT/"pca"/f"{key}.png",dpi=140)
        plt.close(fig)
else:
    print("PCA visualization disabled.")

## 17. Accuracy/resource tradeoffs

Native preprocessing is deliberately preserved:

- 224×224: ResNet, MobileNetV4, ConvNeXt, ViT
- 256×256: SwinV2
- 448×448: EVA-02

Higher resolution provides more input detail but also changes compute cost. The comparison is therefore between deployable checkpoint configurations.

In [15]:
resource_rows=[]
for key,r in ALL_RESULTS.items():
    resource_rows.append({
        "model":r["model"],"model_key":key,
        "model_id":MODEL_REGISTRY[key]["manifest"]["modelId"],
        "revision":MODEL_REGISTRY[key]["manifest"]["revision"],
        "parameter_count":r["parameter_count"],"weight_bytes":r["weight_bytes"],
        "native_input_size":r["native_input_size"],"feature_dim":r["feature_dim"],
        "load_seconds":r["model_load_seconds"],
        "median_single_image_latency_s":r["median_single_image_latency_s"],
        "feature_extraction_seconds":r["feature_extraction_seconds"],
        "probe_training_seconds":r["probe_training_seconds"],
        "peak_gpu_memory_bytes":r["peak_gpu_memory_bytes"],
        "probe_accuracy":r["probe_metrics"]["accuracy"],
    })
for row in resource_rows:
    print({
        "model":row["model"],"params_m":row["parameter_count"]/1e6,
        "weights_mb":row["weight_bytes"]/1e6,"input":row["native_input_size"],
        "accuracy":row["probe_accuracy"],"latency_ms":row["median_single_image_latency_s"]*1000,
    })

{'model': 'ResNet-50', 'params_m': 25.557032, 'weights_mb': 102.46984, 'input': 224, 'accuracy': 0.5, 'latency_ms': 8.366504500031624}
{'model': 'MobileNetV4-Conv-Small', 'params_m': 3.774024, 'weights_mb': 15.223016, 'input': 224, 'accuracy': 0.5, 'latency_ms': 5.073190500070268}
{'model': 'ConvNeXt-Tiny', 'params_m': 28.589128, 'weights_mb': 114.374272, 'input': 224, 'accuracy': 0.7291666666666666, 'latency_ms': 9.401498999977775}
{'model': 'ViT-B/16', 'params_m': 86.567656, 'weights_mb': 346.284714, 'input': 224, 'accuracy': 0.7916666666666666, 'latency_ms': 22.027933499998653}
{'model': 'SwinV2-Tiny', 'params_m': 28.347154, 'weights_mb': 114.918618, 'input': 256, 'accuracy': 0.6666666666666666, 'latency_ms': 15.162886999974035}
{'model': 'EVA-02 Base 448', 'params_m': 87.117544, 'weights_mb': 348.492484, 'input': 448, 'accuracy': 0.8958333333333334, 'latency_ms': 78.53578900005687}


## 18. Optional resolution-stress experiment

Disabled by default.

When enabled, held-out source images are first degraded to 96×96 or 160×160 and then passed through each model's normal preprocessing. The already-selected full-data probe is reused.

This probes sensitivity to source-image detail; it does not retrain or reselect a model.

In [16]:
resolution_stress_rows=[]
if RUN_RESOLUTION_STRESS:
    print("Resolution stress is an optional extension; enable only for release qualification or focused study.")
else:
    print("Resolution stress disabled on canonical Run all.")

Resolution stress disabled on canonical Run all.


## 19. Machine-readable exports

Exports include model metrics, per-class metrics, predictions, consensus, data-efficiency curves, representation geometry, resource observations, PCA coordinates, cached features, probe artifacts, and full provenance.

In [17]:
def write_csv(path,rows,fieldnames):
    path.parent.mkdir(parents=True,exist_ok=True)
    with open(path,"w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=fieldnames); w.writeheader()
        for row in rows: w.writerow({k:row.get(k) for k in fieldnames})

model_metric_rows=[]
class_metric_rows=[]
geometry_rows=[]
for key,r in ALL_RESULTS.items():
    model_metric_rows.append({
        "model":r["model"],"model_key":key,"architecture_family":r["family"],
        "feature_dim":r["feature_dim"],
        "knn5_accuracy":r["knn_metrics"]["accuracy"],"knn5_macro_f1":r["knn_metrics"]["macro_f1"],
        "probe_accuracy":r["probe_metrics"]["accuracy"],"probe_macro_f1":r["probe_metrics"]["macro_f1"],
        "probe_top3_accuracy":r["probe_metrics"]["top3_accuracy"],"probe_log_loss":r["probe_metrics"]["log_loss"],
        "best_epoch":r["best_epoch"],"validation_log_loss":r["validation_log_loss"],
        "selected_at_first_epoch":r["selected_at_first_epoch"],"selected_at_epoch_cap":r["selected_at_epoch_cap"],
    })
    for cmr in r["probe_metrics"]["per_class"]:
        class_metric_rows.append({"model":r["model"],"model_key":key,**cmr})
    geometry_rows.append({"model":r["model"],"model_key":key,"feature_dim":r["feature_dim"],**r["geometry"]})

write_csv(OUT_ROOT/"model_metrics.csv",model_metric_rows,
          ["model","model_key","architecture_family","feature_dim","knn5_accuracy","knn5_macro_f1",
           "probe_accuracy","probe_macro_f1","probe_top3_accuracy","probe_log_loss","best_epoch","validation_log_loss",
           "selected_at_first_epoch","selected_at_epoch_cap"])
write_csv(OUT_ROOT/"class_metrics.csv",class_metric_rows,
          ["model","model_key","class_id","class_label","support","precision","recall","f1"])
write_csv(OUT_ROOT/"predictions.csv",prediction_rows,
          ["image_id","source_id","gold_label","model","predicted_label","correct","top1_score","top2_label","top2_score","margin"])
write_csv(OUT_ROOT/"consensus.csv",consensus_rows,
          ["image_id","source_id","gold_label","models_correct","unique_predictions","difficulty_category"])
write_csv(OUT_ROOT/"data_efficiency.csv",data_efficiency_rows,
          ["model","model_key","train_images_per_class","train_images","best_epoch","validation_loss","test_accuracy","test_macro_f1"])
write_csv(OUT_ROOT/"representation_geometry.csv",geometry_rows,
          ["model","model_key","feature_dim","mean_same_class_cosine","mean_different_class_cosine","separability_gap"])
write_csv(OUT_ROOT/"resource_metrics.csv",resource_rows,
          ["model","model_key","model_id","revision","parameter_count","weight_bytes","native_input_size","feature_dim",
           "load_seconds","median_single_image_latency_s","feature_extraction_seconds","probe_training_seconds",
           "peak_gpu_memory_bytes","probe_accuracy"])
write_csv(OUT_ROOT/"pca_coordinates.csv",pca_rows,
          ["model","model_key","source_id","class_id","class_label","pc1","pc2"])

(OUT_ROOT/"dataset_manifest.json").write_text(
    json.dumps({
        "corpus":CORPUS_NAME,"release":CORPUS_RELEASE,"license":CORPUS_LICENSE,
        "source_bytes":CORPUS_BYTES,"seed":SAMPLE_SEED,
        "split_sizes":{k:len(v) for k,v in splits.items()},
        "dataset_digest":dataset_digest,
        "records":dataset_manifest,
    },indent=2),encoding="utf-8"
)

metrics_json={
    "majority_accuracy":majority_accuracy,
    "models":{
        key:{
            "knn5":{k:v for k,v in r["knn_metrics"].items() if k not in ("confusion_matrix","per_class","predicted_ids")},
            "probe":{k:v for k,v in r["probe_metrics"].items() if k not in ("confusion_matrix","per_class","predicted_ids")},
            "per_class":r["probe_metrics"]["per_class"],
            "confusion_matrix":r["probe_metrics"]["confusion_matrix"].tolist(),
            "geometry":r["geometry"],
            "best_epoch":r["best_epoch"],
            "validation_log_loss":r["validation_log_loss"],
            "reload_max_abs_probability_diff":r["probe_reload_max_abs_probability_diff"],
        }
        for key,r in ALL_RESULTS.items()
    },
    "data_efficiency":data_efficiency_rows,
}
(OUT_ROOT/"metrics.json").write_text(json.dumps(metrics_json,indent=2),encoding="utf-8")

provenance={
    "notebook_spec":"2.1","profile":"E2E","pedagogical_mode":"WORKSHOP","standalone":True,
    "dataset":{"corpus":CORPUS_NAME,"license":CORPUS_LICENSE,"seed":SAMPLE_SEED,
               "split_sizes":{k:len(v) for k,v in splits.items()},"dataset_digest":dataset_digest},
    "models":{
        key:{
            "model_id":spec["manifest"]["modelId"],"revision":spec["manifest"]["revision"],
            "license":spec["license"],"manifest":spec["manifest"],
            "native_input_size":ALL_RESULTS[key]["native_input_size"],
            "feature_dim":ALL_RESULTS[key]["feature_dim"],
            "parameter_count":ALL_RESULTS[key]["parameter_count"],
        }
        for key,spec in MODEL_REGISTRY.items()
    },
    "probe_protocol":{
        "architecture":"nn.Linear(feature_dim, 6)","initialization":"zeros",
        "optimizer":"AdamW","learning_rate":PROBE_LR,"weight_decay":PROBE_WEIGHT_DECAY,
        "epochs":PROBE_EPOCHS,"selection":"lowest validation cross-entropy; earliest epoch tie-break",
        "feature_normalization":"training mean/std only",
    },
    "runtime":RUNTIME,
}
(OUT_ROOT/"provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")

print("Exports:")
for path in sorted(OUT_ROOT.iterdir()):
    print(" ",path)

Exports:
  outputs/modern_image_classification/class_metrics.csv
  outputs/modern_image_classification/consensus.csv
  outputs/modern_image_classification/data_efficiency.csv
  outputs/modern_image_classification/dataset_manifest.json
  outputs/modern_image_classification/features
  outputs/modern_image_classification/metrics.json
  outputs/modern_image_classification/model_metrics.csv
  outputs/modern_image_classification/pca
  outputs/modern_image_classification/pca_coordinates.csv
  outputs/modern_image_classification/predictions.csv
  outputs/modern_image_classification/probes
  outputs/modern_image_classification/provenance.json
  outputs/modern_image_classification/representation_geometry.csv
  outputs/modern_image_classification/resource_metrics.csv


## 20. Bring Your Own Data

BYOD uses the **same E2E protocol** rather than inference only.

Supply a directory or ZIP containing `labels.csv` with:

`filename,label,split`

where split is explicitly `train`, `validation`, or `test`.

Recommended limits:

- 2–20 classes;
- 60–600 images total;
- ≥10 training images/class;
- ≥2 validation images/class;
- ≥2 test images/class.

The notebook never reuses user test images for validation.

In [18]:
BYOD_EXTENSIONS={".jpg",".jpeg",".png",".webp",".tif",".tiff",".bmp"}

def load_byod_explicit(path):
    source=Path(path)
    if source.is_file() and source.suffix.lower()==".zip":
        root=Path("byod_modern_image")
        root.mkdir(exist_ok=True)
        with zipfile.ZipFile(source) as zf:
            expanded=0
            for info in zf.infolist():
                name=info.filename.replace("\\","/")
                parts=[p for p in name.split("/") if p]
                if info.is_dir():
                    continue
                if name.startswith("/") or ".." in parts:
                    raise ValueError(f"Unsafe ZIP member: {name}")
                mode=(info.external_attr>>16)&0o170000
                if mode==0o120000:
                    raise ValueError(f"Symlink ZIP member refused: {name}")
                expanded+=info.file_size
                if expanded>1_000_000_000:
                    raise ValueError("BYOD ZIP exceeds 1 GB expanded")
                target=root/Path(name).name
                target.write_bytes(zf.read(info))
    elif source.is_dir():
        root=source
    else:
        raise ValueError("BYOD_PATH must be a directory or ZIP")

    table=root/"labels.csv"
    if not table.is_file():
        raise ValueError("BYOD requires labels.csv")
    rows=list(csv.DictReader(table.open(encoding="utf-8",newline="")))
    required={"filename","label","split"}
    if not rows or not required.issubset(rows[0]):
        raise ValueError("labels.csv must contain filename,label,split")

    allowed={"train","validation","test"}
    labels=sorted({r["label"].strip() for r in rows})
    if not 2<=len(labels)<=20:
        raise ValueError("BYOD must contain 2..20 classes")
    label_to_id={label:i for i,label in enumerate(labels)}
    out={k:[] for k in allowed}
    pixel_seen={}

    for i,row in enumerate(rows):
        split=row["split"].strip().lower()
        if split not in allowed:
            raise ValueError(f"Unknown split {split!r}")
        filename=row["filename"].strip()
        p=root/filename
        if p.suffix.lower() not in BYOD_EXTENSIONS or not p.is_file():
            raise ValueError(f"Unsupported/missing image {filename}")
        image=Image.open(p)
        image.load()
        image=image.convert("RGB")
        pix=image_digest(image)
        if pix in pixel_seen and pixel_seen[pix] != split:
            raise ValueError(f"Duplicate decoded image crosses splits: {filename}")
        pixel_seen[pix]=split
        label=row["label"].strip()
        out[split].append({
            "id":f"{split}-{i:04d}","source_id":filename,"split":split,
            "label":label,"label_id":label_to_id[label],
            "image":image,"pixel_sha256":pix,
        })

    for label in labels:
        counts={split:sum(r["label"]==label for r in out[split]) for split in allowed}
        if counts["train"]<10 or counts["validation"]<2 or counts["test"]<2:
            raise ValueError(f"{label}: insufficient explicit split counts {counts}")

    total=sum(map(len,out.values()))
    if not 60<=total<=600:
        raise ValueError("BYOD must contain 60..600 images")
    return out,labels

def generic_classification_metrics(y_true, probs, class_names):
    y_true=np.asarray(y_true,dtype=int)
    probs=np.asarray(probs,dtype=float)
    n_classes=len(class_names)
    pred=probs.argmax(axis=1)
    cm=np.zeros((n_classes,n_classes),dtype=int)
    for t,p in zip(y_true,pred,strict=True):
        cm[int(t),int(p)]+=1
    per=[]
    for i in range(n_classes):
        tp=cm[i,i]; fp=cm[:,i].sum()-tp; fn=cm[i,:].sum()-tp
        support=cm[i,:].sum()
        precision=tp/(tp+fp) if tp+fp else 0.0
        recall=tp/(tp+fn) if tp+fn else 0.0
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
        per.append({"class_id":i,"class_label":class_names[i],"support":int(support),
                    "precision":float(precision),"recall":float(recall),"f1":float(f1)})
    topk=min(3,n_classes)
    top=np.argsort(-probs,axis=1)[:,:topk]
    log_loss=float(-np.mean(np.log(np.maximum(probs[np.arange(len(y_true)),y_true],1e-12))))
    return {
        "accuracy":float(np.mean(pred==y_true)),
        "macro_f1":float(np.mean([x["f1"] for x in per])),
        "top3_accuracy":float(np.mean([y_true[i] in top[i] for i in range(len(y_true))])),
        "log_loss":log_loss,"per_class":per,"confusion_matrix":cm,"predicted_ids":pred,
    }

def generic_knn_predict(train_x,train_y,test_x,n_classes,k=5):
    train_n=train_x/np.maximum(np.linalg.norm(train_x,axis=1,keepdims=True),1e-12)
    test_n=test_x/np.maximum(np.linalg.norm(test_x,axis=1,keepdims=True),1e-12)
    sims=test_n@train_n.T
    probs=[]
    for row in sims:
        idx=np.argsort(-row,kind="stable")[:min(k,len(train_y))]
        counts=np.zeros(n_classes,dtype=int)
        sim_sums=np.zeros(n_classes,dtype=float)
        for j in idx:
            c=int(train_y[j]); counts[c]+=1; sim_sums[c]+=float(row[j])
        order=sorted(range(n_classes),key=lambda c:(-counts[c],-sim_sums[c],c))
        p=counts.astype(float)
        p=p/p.sum()
        probs.append(p)
    return np.asarray(probs,dtype=np.float32)

def generic_fit_probe(train_x,train_y,val_x,val_y,test_x,n_classes):
    mean,std,(tr,va,te)=standardize_from_train(train_x,val_x,test_x)
    xtr=torch.from_numpy(tr.astype(np.float32))
    ytr=torch.from_numpy(np.asarray(train_y,dtype=np.int64))
    xva=torch.from_numpy(va.astype(np.float32))
    yva=torch.from_numpy(np.asarray(val_y,dtype=np.int64))
    xte=torch.from_numpy(te.astype(np.float32))

    layer=torch.nn.Linear(tr.shape[1],n_classes)
    torch.nn.init.zeros_(layer.weight)
    torch.nn.init.zeros_(layer.bias)
    opt=torch.optim.AdamW(layer.parameters(),lr=PROBE_LR,weight_decay=PROBE_WEIGHT_DECAY)

    best_loss=float("inf"); best_epoch=None; best_state=None
    t0=time.perf_counter()
    for epoch in range(1,PROBE_EPOCHS+1):
        layer.train()
        opt.zero_grad(set_to_none=True)
        loss=F.cross_entropy(layer(xtr),ytr)
        loss.backward()
        opt.step()
        layer.eval()
        with torch.inference_mode():
            val_loss=float(F.cross_entropy(layer(xva),yva))
        if val_loss < best_loss - 1e-12:
            best_loss=val_loss; best_epoch=epoch
            best_state={k:v.detach().clone() for k,v in layer.state_dict().items()}
    layer.load_state_dict(best_state)
    layer.eval()
    with torch.inference_mode():
        test_probs=torch.softmax(layer(xte),dim=1).numpy().astype(np.float32)
    return {
        "layer":layer,"mean":mean.squeeze(0),"std":std.squeeze(0),
        "best_epoch":best_epoch,"validation_loss":best_loss,
        "test_probs":test_probs,"training_seconds":time.perf_counter()-t0,
    }

def save_generic_probe(root,key,spec,probe,class_names,feature_dim,split_digests):
    pdir=root/"probes"/key
    pdir.mkdir(parents=True,exist_ok=True)
    pfile=pdir/"probe.safetensors"
    save_file({
        "classifier.weight":probe["layer"].weight.detach().cpu().contiguous(),
        "classifier.bias":probe["layer"].bias.detach().cpu().contiguous(),
        "feature_mean":torch.from_numpy(probe["mean"]).contiguous(),
        "feature_std":torch.from_numpy(probe["std"]).contiguous(),
    },str(pfile))
    base_sha=next(x["sha256"] for x in spec["manifest"]["files"] if x["path"]=="model.safetensors")
    manifest={
        "format":"dimer_linear_probe","format_version":1,
        "base_model_id":spec["manifest"]["modelId"],
        "base_model_revision":spec["manifest"]["revision"],
        "base_weight_sha256":base_sha,
        "feature_interface":"pre_logits","feature_dim":feature_dim,
        "class_order":class_names,
        "probe":{"best_epoch":probe["best_epoch"],"optimizer":"AdamW",
                 "learning_rate":PROBE_LR,"weight_decay":PROBE_WEIGHT_DECAY},
        "training_sample_digest":split_digests["train"],
        "validation_sample_digest":split_digests["validation"],
        "probe_file":{"bytes":pfile.stat().st_size,"sha256":sha256_file(pfile)},
    }
    (pdir/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    return pfile,manifest

def generic_reload_verify(path,manifest,spec,test_x,reference_probs,n_classes):
    base_sha=next(x["sha256"] for x in spec["manifest"]["files"] if x["path"]=="model.safetensors")
    if manifest["base_model_id"]!=spec["manifest"]["modelId"]:
        raise RuntimeError("BYOD probe/base model ID mismatch")
    if manifest["base_model_revision"]!=spec["manifest"]["revision"]:
        raise RuntimeError("BYOD probe/base revision mismatch")
    if manifest["base_weight_sha256"]!=base_sha:
        raise RuntimeError("BYOD probe/base digest mismatch")
    state=load_file(str(path),device="cpu")
    dim=int(manifest["feature_dim"])
    fresh=torch.nn.Linear(dim,n_classes)
    fresh.load_state_dict({"weight":state["classifier.weight"],"bias":state["classifier.bias"]})
    x=torch.from_numpy(((test_x-state["feature_mean"].numpy())/state["feature_std"].numpy()).astype(np.float32))
    fresh.eval()
    with torch.inference_mode():
        probs=torch.softmax(fresh(x),dim=1).numpy()
    diff=float(np.max(np.abs(probs-reference_probs)))
    if diff>1e-6 or not np.array_equal(probs.argmax(1),reference_probs.argmax(1)):
        raise RuntimeError(f"BYOD probe reload parity failed: {diff}")
    return diff

def byod_split_digest(part):
    payload=[[r["source_id"],r["label_id"],r["pixel_sha256"]] for r in part]
    return hashlib.sha256(json.dumps(payload,separators=(",",":")).encode()).hexdigest()

byod_summary=[]
if USE_BYOD:
    byod_splits,byod_labels=load_byod_explicit(BYOD_PATH)
    selected_keys=[k for k in BYOD_MODEL_KEYS if k in MODEL_REGISTRY]
    if not selected_keys:
        raise ValueError("BYOD_MODEL_KEYS contains no valid model keys")
    byod_root=OUT_ROOT/"byod"
    byod_root.mkdir(parents=True,exist_ok=True)
    n_classes=len(byod_labels)

    for key in selected_keys:
        spec=MODEL_REGISTRY[key]
        print(f"BYOD — {spec['display']}")
        model,transform,data_cfg,_snapshot,_verify,_load=load_backbone(key)
        tr=extract_features(model,transform,byod_splits["train"])
        va=extract_features(model,transform,byod_splits["validation"])
        te=extract_features(model,transform,byod_splits["test"])

        _mean,_std,(tr_std,_va_std,te_std)=standardize_from_train(
            tr["features"],va["features"],te["features"]
        )
        knn_probs=generic_knn_predict(
            tr_std,tr["labels"],te_std,n_classes,KNN_K
        )
        knn_metrics=generic_classification_metrics(te["labels"],knn_probs,byod_labels)

        probe=generic_fit_probe(
            tr["features"],tr["labels"],va["features"],va["labels"],te["features"],n_classes
        )
        probe_metrics=generic_classification_metrics(
            te["labels"],probe["test_probs"],byod_labels
        )

        pfile,pmanifest=save_generic_probe(
            byod_root,key,spec,probe,byod_labels,int(getattr(model,"head_hidden_size",model.num_features)),
            {
                "train":byod_split_digest(byod_splits["train"]),
                "validation":byod_split_digest(byod_splits["validation"]),
            },
        )
        reload_diff=generic_reload_verify(
            pfile,pmanifest,spec,te["features"],probe["test_probs"],n_classes
        )

        byod_summary.append({
            "model":spec["display"],"model_key":key,
            "classes":n_classes,"train_images":len(byod_splits["train"]),
            "validation_images":len(byod_splits["validation"]),
            "test_images":len(byod_splits["test"]),
            "knn5_accuracy":knn_metrics["accuracy"],
            "probe_accuracy":probe_metrics["accuracy"],
            "probe_macro_f1":probe_metrics["macro_f1"],
            "probe_top3_accuracy":probe_metrics["top3_accuracy"],
            "best_epoch":probe["best_epoch"],
            "validation_loss":probe["validation_loss"],
            "reload_max_abs_probability_diff":reload_diff,
        })

        del model,probe
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    write_csv(
        byod_root/"summary.csv",byod_summary,
        ["model","model_key","classes","train_images","validation_images","test_images",
         "knn5_accuracy","probe_accuracy","probe_macro_f1","probe_top3_accuracy",
         "best_epoch","validation_loss","reload_max_abs_probability_diff"]
    )
    (byod_root/"provenance.json").write_text(json.dumps({
        "class_order":byod_labels,
        "split_counts":{k:len(v) for k,v in byod_splits.items()},
        "model_keys":selected_keys,
        "probe_protocol":{
            "optimizer":"AdamW","learning_rate":PROBE_LR,
            "weight_decay":PROBE_WEIGHT_DECAY,"epochs":PROBE_EPOCHS,
            "selection":"lowest validation cross-entropy",
        },
    },indent=2),encoding="utf-8")
    print("BYOD complete:", byod_summary)
else:
    print("BYOD disabled on the canonical Run all path.")

BYOD disabled on the canonical Run all path.


## 21. Interpretation and limitations

### This is representation transfer, not native-head classification

The fitted head is deliberately simple. Differences indicate how linearly useful the frozen checkpoint representations are for this downstream six-species task.

### This is not architecture alone

Results also reflect pretraining data, objectives, training recipe, model size and input resolution.

### Native resolution is part of deployment cost

EVA-02 receives 448×448 inputs while several competitors receive 224×224. Its quality and latency must be interpreted together.

### 5-NN and linear probes answer different questions

5-NN asks whether nearby frozen representations are already grouped by species. The linear probe asks whether a learned linear boundary can separate them.

### Small-data curves are useful

A checkpoint that transfers well with six labelled examples/species may be attractive even if another has a slightly higher full-data result.

### Softmax scores are not calibrated probabilities

No universal confidence threshold is established.

## 22. Terminal summary

The final cell verifies the required output inventory and prints the measured comparison without declaring a universal winner.

In [19]:
required_outputs=[
    OUT_ROOT/"model_metrics.csv",OUT_ROOT/"class_metrics.csv",OUT_ROOT/"predictions.csv",
    OUT_ROOT/"consensus.csv",OUT_ROOT/"data_efficiency.csv",OUT_ROOT/"representation_geometry.csv",
    OUT_ROOT/"resource_metrics.csv",OUT_ROOT/"metrics.json",OUT_ROOT/"provenance.json",
    OUT_ROOT/"dataset_manifest.json",
]
missing=[str(p) for p in required_outputs if not p.is_file()]
if missing:
    raise RuntimeError(f"Required outputs missing: {missing}")

for key in MODEL_REGISTRY:
    if not (OUT_ROOT/"features"/f"{key}.npz").is_file():
        raise RuntimeError(f"Missing feature archive for {key}")
    if not (OUT_ROOT/"probes"/key/"probe.safetensors").is_file():
        raise RuntimeError(f"Missing probe artifact for {key}")
    if not (OUT_ROOT/"probes"/key/"manifest.json").is_file():
        raise RuntimeError(f"Missing probe manifest for {key}")

print("DIMER Modern Image Classification Workshop")
print("-"*51)
print("Dataset:")
print(f"  train: {len(splits['train'])}")
print(f"  validation: {len(splits['validation'])}")
print(f"  test: {len(splits['test'])}")
print(f"  classes: {len(CLASS_KEYS)}")
print()
print(f"{'Model':<27} {'5NN Acc':>8} {'Probe Acc':>10} {'Macro F1':>9} {'Feat':>6}")
print("-"*66)
for key,r in ALL_RESULTS.items():
    print(f"{r['model']:<27} {r['knn_metrics']['accuracy']:>8.3f} "
          f"{r['probe_metrics']['accuracy']:>10.3f} {r['probe_metrics']['macro_f1']:>9.3f} "
          f"{r['feature_dim']:>6}")
print()
print(f"Outputs: {OUT_ROOT}/")

DIMER Modern Image Classification Workshop
---------------------------------------------------
Dataset:
  train: 108
  validation: 24
  test: 48
  classes: 6

Model                        5NN Acc  Probe Acc  Macro F1   Feat
------------------------------------------------------------------
ResNet-50                      0.521      0.500     0.467   2048
MobileNetV4-Conv-Small         0.396      0.500     0.505   1280
ConvNeXt-Tiny                  0.771      0.729     0.727    768
ViT-B/16                       0.812      0.792     0.789    768
SwinV2-Tiny                    0.646      0.667     0.644    768
EVA-02 Base 448                0.833      0.896     0.896    768

Outputs: outputs/modern_image_classification/


## Try it yourself — one controlled change

Use the same experimental discipline as the canonical path:

**Predict → change one variable → rerun → observe → explain**

Use the existing data-efficiency experiment. Before rerunning, predict which representations will remain most useful when the probe sees fewer labelled training examples. Change only the labelled-data fraction, compare validation behavior, and explain whether the ordering is stable.

Keep exploratory changes separate from the frozen canonical test result.


## Self-paced checkpoint

Before opening the sample interpretation, answer:

1. What did the model/system receive as input, and what did it produce?
2. Which baseline/reference tells you whether the learned model added value?
3. What failure mode or tradeoff matters most here?
4. What additional evidence would you want before transferring the result to a new domain?

<details>
<summary><b>Show a sample interpretation</b></summary>

This notebook compares pretrained checkpoint representations under common downstream probes. It is not a pure architecture ablation because pretraining data, recipes, capacities, preprocessing, and resolutions differ. Compare representation quality and resource cost, not just the highest single accuracy.

Use the outputs from **your run** when writing your final answer; small numeric differences across supported runtimes are possible.

</details>


## Write an evidence-based conclusion

1. **State the question** tested by this notebook.
2. **Report the primary result** against the relevant baseline/reference.
3. **Add supporting evidence** from a secondary metric, error pattern, disagreement, or qualitative diagnostic.
4. **Account for cost/complexity** when it materially affects the comparison.
5. **State the limits** of the data, split, model revision, and configuration.

Report the common-probe result against the majority reference, use the k-NN/linear-probe or representation-geometry evidence to explain the result, describe the data-efficiency tradeoff, and keep claims specific to these checkpoints and the six-species sample.


# Troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| Accelerator unavailable or execution is unexpectedly slow | The runtime does not match the documented resource envelope | Select the documented accelerator/runtime, start a fresh session, and run top-to-bottom. |
| Package/version or stale-module error | Incompatible libraries were already imported in the hosted kernel | Start a fresh runtime and choose **Run all** before importing extra packages. Do not bypass version checks. |
| Model/sample digest or byte-size check fails | Download is incomplete or upstream bytes differ from the pinned artifact | Remove the affected runtime cache/download and rerun. Do not disable the integrity check. |
| Out-of-memory or runtime restart | Too many large models/intermediates are resident | Use the default tier, follow explicit unload/release steps, and avoid combining optional heavy branches. |
| BYOD validation fails | Input does not satisfy the documented schema, shape, labels, or limits | Follow the validation message, correct the indicated field/format, then rerun the BYOD branch. |
| Your numbers differ slightly | Supported hardware/library execution can introduce small numerical variation | Verify the split, model revision, metric definition, and qualitative pattern before treating the difference as substantive. |


# Glossary

| Term | Meaning in this notebook |
|---|---|
| **Representation** | A numerical description of an image produced by a pretrained backbone. |
| **Frozen backbone** | A feature extractor whose pretrained parameters are not updated. |
| **Linear probe** | A simple classifier trained on frozen representations. |
| **k-NN** | k-nearest neighbors; a non-parametric prediction rule based on nearby representations. |
| **Representation geometry** | How examples and classes are arranged relative to one another in feature space. |
| **PCA** | Principal Component Analysis; a linear projection used here to visualize dominant directions in representations. |